# Laboratorio de Análisis Numérico — Reto: del píxel al volumen
## Fase 1 — Motor numérico validado sobre un envase de proporciones conocidas

---

### Qué es este notebook y qué NO es todavía

El reto completo exige la cadena:

```
captura → imagen → escala → datos → modelo → integración → volumen → validación
```

Este notebook resuelve **la segunda mitad** de esa cadena:

```
                          ┌─────── ESTE NOTEBOOK ────────┐
captura → imagen → escala → datos → modelo → integración → volumen → validación
└──── fase 2 (cámara) ───┘
```

**Decisión de ingeniería — por qué empezamos por aquí y no por la cámara.**

Cuando la foto real dé un volumen con 9 % de error, ese error es la suma de cuatro
contribuciones independientes:

| Fuente de error | ¿Se puede medir por separado? |
|---|---|
| Método de integración (Δz, orden) | **Sí — aquí, contra un valor exacto** |
| Escalamiento píxel → cm | Solo con marcador de referencia |
| Extracción del contorno | Solo con imagen real |
| Proyección de la fotografía | Solo con cámara real |

Si mezclamos todo desde el principio, el error es **un número opaco**. Depurar un
sistema así es adivinar. Aquí eliminamos la primera fuente por completo: usamos un
envase **definido matemáticamente**, cuyo volumen exacto conocemos por fórmula cerrada.
Cualquier discrepancia que aparezca es, con certeza, del integrador.

Cuando esta fase termine sabremos, con números, **cuál es el piso de error del motor
numérico**. Todo lo que exceda ese piso en la fase de cámara vendrá de la imagen.

---

### Mapa: qué celda cubre qué parte del PDF del laboratorio

| Sección del notebook | Etapa / requisito del PDF |
|---|---|
| 1. Marco matemático | §5 Contexto matemático |
| 2. Modelo del envase | §7 Etapa 1 (definición del problema) |
| 3. Volumen de referencia exacto | §7 Etapa 6 (valor de referencia independiente) |
| 4. Tabla de datos (zᵢ, rᵢ, rᵢ², πrᵢ²) | §7 Etapa 4 (construcción de datos) |
| 5. Los cuatro métodos | §6 Métodos numéricos mínimos |
| 6. Validación de condiciones | §8 "mensajes claros cuando los datos no cumplen" |
| 7. Tabla comparativa | §7 Etapa 5 + §8 (reporte de Δz y parámetros) |
| 8. Estudio de convergencia | §6 "Efecto de Δz y del número de puntos" |
| 9. Gráficas | §8 "representación gráfica del perfil y de πr²" |
| 10. Exportación | §8 "exportación de datos/resultados para el informe" |
| 11. Discusión de errores | §9 MVP "explicar las principales fuentes de error" |

---

### Cómo usar este notebook

Ejecuta las celdas **en orden** (`Entorno de ejecución → Ejecutar todo`, o `Ctrl+F9`).
Cada celda de código va precedida de una celda de texto que explica **qué hace**, y
seguida de una explicación de **cómo leer lo que imprime**.

No requiere instalar nada: Colab ya trae `numpy`, `scipy`, `pandas` y `matplotlib`.

---
## 0. Verificación del entorno

**Qué hace esta celda:** importa las librerías y muestra sus versiones.

**Por qué importa:** los resultados numéricos de un laboratorio deben ser
*reproducibles*. Si dentro de seis meses alguien obtiene otro número, lo primero que
se revisa es la versión de las librerías. Dejar constancia de ellas es parte de la
buena práctica científica, no un adorno.

`np.set_printoptions` y la configuración de `pandas` fijan la precisión de
visualización a 6 decimales. **Ojo con la distinción:** esto cambia lo que *ves*, no
lo que se *calcula*. Internamente todo opera en `float64` (doble precisión IEEE-754,
≈ 15-16 dígitos decimales significativos).

In [ ]:
import time
import math
from dataclasses import dataclass, field
from typing import Callable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import integrate, interpolate

# --- Configuración de visualización -------------------------------------
# Afecta SOLO a la impresión en pantalla; los cálculos siguen en float64.
np.set_printoptions(precision=6, suppress=True, linewidth=110)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")
pd.set_option("display.max_rows", 60)
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print("=" * 66)
print("ENTORNO DE EJECUCIÓN")
print("=" * 66)
print(f"  NumPy       : {np.__version__}")
print(f"  SciPy       : {scipy.__version__}")
print(f"  pandas      : {pd.__version__}")
print(f"  Precisión   : float64  (eps = {np.finfo(float).eps:.3e})")
print("=" * 66)
print("\nListo. Ejecuta las celdas en orden.")

**Cómo leer el resultado:** debes ver las cuatro versiones y el valor de
`eps ≈ 2.220e-16`.

Ese `eps` es el **épsilon de máquina**: el menor número tal que `1.0 + eps != 1.0` en
doble precisión. Es la cota inferior absoluta del error de cualquier cálculo en este
notebook. Ningún método de integración, por sofisticado que sea, puede darte un
resultado más exacto que ~1e-16 relativo. Lo vas a ver aparecer explícitamente en la
Sección 8: la curva de error deja de bajar y se aplana en torno a 1e-15 — ese suelo
**no es un defecto del método, es el límite de la aritmética del computador**.

---
## 1. Marco matemático: el sólido de revolución

### 1.1 De dónde sale la integral

Modelamos el envase como un **sólido de revolución**: la figura que se obtiene al girar
una curva plana 360° alrededor de un eje. Llamamos $z$ a la coordenada a lo largo del
eje (la altura) y $r(z)$ al radio del envase a esa altura.

La construcción es la del **método de discos**. Cortamos el sólido con planos
perpendiculares al eje, separados una distancia infinitesimal $dz$. Cada rebanada es
(aproximadamente) un cilindro muy delgado:

$$dV = \underbrace{\pi\,[r(z)]^2}_{\text{área del disco}} \cdot \underbrace{dz}_{\text{espesor}}$$

Sumando todas las rebanadas entre la base $a$ y la boca $b$:

$$\boxed{\;V \;=\; \pi \int_{a}^{b} [r(z)]^{2}\, dz\;}$$

que es exactamente la expresión de la §5 del enunciado.

### 1.2 El supuesto que estamos haciendo (y que hay que declarar)

Esta fórmula **presupone simetría de revolución**: que toda sección transversal del
envase es un círculo. Es exacto para una lata o una botella PET estándar; es **falso**
para un tetrapak (sección rectangular) o un frasco ovalado.

Es el supuesto central del laboratorio y debe ir declarado en el informe. En la fase de
cámara podremos incluso *cuantificar* cuánto se viola, comparando el radio medido a la
izquierda del eje contra el medido a la derecha.

### 1.3 Por qué separamos la integral del factor π — una decisión de diseño

Dentro del código no vamos a integrar $\pi r^2$ directamente. Vamos a integrar

$$I \;=\; \int_a^b [r(z)]^2\,dz \qquad\text{y luego}\qquad V = \pi\, I$$

Tres razones:

1. **Separación de responsabilidades.** Los métodos de cuadratura son operadores
   genéricos sobre una función tabulada; no tienen por qué saber nada de geometría.
   Mantenerlos ignorantes de $\pi$ los hace reutilizables y testeables contra
   integrales estándar.
2. **Trazabilidad.** El PDF (§7 Etapa 5) pide reportar *"valor de la integral estimada"*
   **y** *"volumen estimado"* como dos cantidades distintas. Con esta separación son
   genuinamente dos números, no el mismo repetido.
3. **Análisis de error limpio.** El error relativo es idéntico en $I$ y en $V$, porque
   $\pi$ es una constante multiplicativa exacta. Así ningún lector puede sospechar que
   el factor $\pi$ introduce error. No lo introduce, y queda demostrado por construcción.

### 1.4 La consecuencia más importante de todo el laboratorio

Observa que el radio entra **al cuadrado**. Si medimos el radio con un error relativo
$\varepsilon_r$, propagándolo:

$$\frac{\Delta V}{V} \;=\; 2\,\frac{\Delta r}{r} \;=\; 2\,\varepsilon_r$$

**El error del radio se duplica en el volumen.** Como el radio vendrá de una conversión
píxel→cm, esto fija el presupuesto de error de todo el proyecto:

| Error en la escala px→cm | Error resultante en el volumen |
|---|---|
| 1 % | 2 % ✅ |
| 2.5 % | 5 % ⚠️ (justo en el límite del reto) |
| 5 % | 10 % ❌ |

El criterio del 5 % de la §10 del PDF se juega, por lo tanto, **en la calibración de la
escala, no en la elección del método de integración**. Lo demostraremos con números en
la Sección 8: la diferencia entre Simpson y Gauss sobre estos datos será de órdenes de
magnitud *menor* que cualquier error de escala realista.

---
## 2. Modelo del envase: perfil paramétrico de proporciones conocidas

### 2.1 La idea

En lugar de una foto, definimos el envase como una **lista de tramos**. Cada tramo va de
una altura $z_i$ a otra $z_{i+1}$, y el radio varía **linealmente** entre $r_i$ y $r_{i+1}$.

Geométricamente cada tramo es un **tronco de cono** (un cono al que se le corta la
punta). Si $r_i = r_{i+1}$, el tronco degenera en un **cilindro**.

```
   r(z)
    │                    ╭──────╮  boca / rosca
    │                    │      │
    │                    │      │  cuello  (cilindro)
    │                   ╱        ╲
    │                  ╱          ╲ hombro  (tronco de cono)
    │      ┌──────────╯            ╰──────────┐
    │      │                                  │  cuerpo  (cilindro)
    │      │                                  │
    │       ╲                                ╱   base    (tronco de cono)
    └────────┴──────────────────────────────┴──────────────► z
             a                                            b
```

### 2.2 Por qué precisamente este modelo — la razón es el valor de referencia

El PDF (§7 Etapa 6) exige *"un envase cuya geometría permita obtener un volumen de
referencia independiente"*. Un perfil lineal por tramos lo da **en forma cerrada**: el
volumen de un tronco de cono de altura $h$ y radios $R_1, R_2$ es

$$V_{\text{tronco}} \;=\; \frac{\pi h}{3}\left(R_1^{2} + R_1 R_2 + R_2^{2}\right)$$

El volumen total es la suma sobre los tramos. Ese número es **exacto** — no es otra
aproximación numérica con la que comparar, es *la respuesta*. Podemos medir el error de
cada método hasta el último dígito significativo.

### 2.3 El detalle fino: los puntos de quiebre, y por qué los ponemos a propósito

El perfil es **continuo** ($C^0$) pero **no derivable** ($C^1$) en las uniones entre
tramos: ahí la pendiente cambia de golpe. Esto no es un descuido del modelo, es
**deliberado y realista**: una botella PET real tiene exactamente esas aristas, en el
pie y en el arranque del hombro.

Y tiene una consecuencia numérica de primer orden, que es una de las lecciones centrales
de este laboratorio:

> Las cotas de error de Simpson, Newton–Cotes y Gauss suponen que el integrando es
> **suave** (derivadas continuas hasta cierto orden). En una esquina esa hipótesis se
> rompe, y los métodos de orden alto **pierden su orden teórico**.

En la Sección 8 lo veremos ocurrir con datos, y después lo confirmaremos por contraste
con un envase deliberadamente suave. Es exactamente la clase de discusión que pide la
§7 Etapa 6 del enunciado: distinguir si la diferencia se debe al método, a la
discretización o al modelo.

### 2.4 Estructuras de datos

**Qué hace esta celda:** define los tipos `Tramo` y `PerfilEnvase`.

Usamos `@dataclass(frozen=True)` — objetos **inmutables**. Una vez construido el perfil,
ninguna celda posterior puede modificarlo por accidente. En un notebook, donde las
celdas se ejecutan en cualquier orden y el estado persiste, la mutación accidental es
*la* fuente clásica de resultados irreproducibles. La inmutabilidad la elimina de raíz.

Métodos que expone `PerfilEnvase`:

| Método | Qué devuelve |
|---|---|
| `radio(z)` | el radio $r(z)$, vectorizado sobre arrays de NumPy |
| `volumen_exacto()` | el volumen analítico en cm³ (fórmula del tronco) |
| `puntos_de_quiebre()` | las alturas donde el perfil no es derivable |
| `a`, `b`, `altura` | extremos e intervalo de integración |

In [ ]:
@dataclass(frozen=True)
class Tramo:
    """Un tramo del envase con radio que varía linealmente con la altura.

    Geométricamente es un tronco de cono (cilindro si r_ini == r_fin).

    Atributos
    ---------
    z_ini, z_fin : float   Alturas inicial y final del tramo [cm].
    r_ini, r_fin : float   Radios en esas alturas [cm].
    nombre       : str     Etiqueta descriptiva (p. ej. "cuerpo").
    """
    z_ini: float
    z_fin: float
    r_ini: float
    r_fin: float
    nombre: str = ""

    def __post_init__(self) -> None:
        # Validación temprana: un tramo inválido detectado aquí produce un error
        # comprensible. Detectado 200 líneas más abajo produce un NaN inexplicable.
        if self.z_fin <= self.z_ini:
            raise ValueError(
                f"Tramo '{self.nombre}': z_fin ({self.z_fin}) debe ser > "
                f"z_ini ({self.z_ini}). Los tramos van de menor a mayor altura."
            )
        if self.r_ini < 0 or self.r_fin < 0:
            raise ValueError(
                f"Tramo '{self.nombre}': los radios no pueden ser negativos "
                f"(r_ini={self.r_ini}, r_fin={self.r_fin})."
            )

    @property
    def altura(self) -> float:
        """Altura h del tramo [cm]."""
        return self.z_fin - self.z_ini

    def volumen_exacto(self) -> float:
        """Volumen analítico del tronco de cono [cm^3].

            V = (pi*h/3) * (R1^2 + R1*R2 + R2^2)

        Para R1 == R2 == R se reduce a pi*R^2*h, el cilindro. Es útil verificarlo
        mentalmente: (pi*h/3)(R^2+R^2+R^2) = (pi*h/3)(3R^2) = pi*R^2*h.
        """
        R1, R2 = self.r_ini, self.r_fin
        return (math.pi * self.altura / 3.0) * (R1 * R1 + R1 * R2 + R2 * R2)

    def radio(self, z: np.ndarray) -> np.ndarray:
        """Radio interpolado linealmente dentro del tramo [cm]."""
        t = (z - self.z_ini) / self.altura           # parámetro normalizado en [0,1]
        return self.r_ini + t * (self.r_fin - self.r_ini)


@dataclass(frozen=True)
class PerfilEnvase:
    """Envase completo como una secuencia contigua de tramos.

    Es el 'gemelo digital' del envase: sustituye a la fotografía en esta fase.
    """
    tramos: tuple[Tramo, ...]
    nombre: str = "envase"
    capacidad_nominal_ml: float | None = None   # solo informativo (etiqueta comercial)

    def __post_init__(self) -> None:
        if not self.tramos:
            raise ValueError("El perfil debe tener al menos un tramo.")
        # Los tramos deben encadenarse sin huecos ni solapamientos: el final de uno
        # es el inicio del siguiente. Un hueco produciría un volumen silenciosamente
        # incorrecto, que es el peor tipo de error: no falla, solo miente.
        for t1, t2 in zip(self.tramos, self.tramos[1:]):
            if not math.isclose(t1.z_fin, t2.z_ini, abs_tol=1e-12):
                raise ValueError(
                    f"Discontinuidad entre '{t1.nombre}' y '{t2.nombre}': "
                    f"{t1.nombre} termina en z={t1.z_fin} pero {t2.nombre} "
                    f"empieza en z={t2.z_ini}. Deben coincidir."
                )

    @property
    def a(self) -> float:
        """Límite inferior de integración = base del envase [cm]."""
        return self.tramos[0].z_ini

    @property
    def b(self) -> float:
        """Límite superior de integración = boca del envase [cm]."""
        return self.tramos[-1].z_fin

    @property
    def altura(self) -> float:
        """Altura total b - a [cm]."""
        return self.b - self.a

    def radio(self, z: float | np.ndarray) -> np.ndarray:
        """Radio r(z) evaluado en uno o muchos puntos. Vectorizado.

        Implementación: np.piecewise selecciona, para cada z, el tramo al que
        pertenece y le aplica su interpolación lineal. Vectorizar importa porque en
        la Sección 8 evaluaremos r(z) cientos de miles de veces.
        """
        z_arr = np.atleast_1d(np.asarray(z, dtype=float))
        # clip evita que un z fuera de [a,b] por redondeo produzca basura;
        # lo satura al borde más cercano, que es el comportamiento físico correcto.
        z_arr = np.clip(z_arr, self.a, self.b)
        condiciones = [
            (z_arr >= t.z_ini) & (z_arr <= t.z_fin) for t in self.tramos
        ]
        funciones = [
            (lambda zz, tr=t: tr.radio(zz)) for t in self.tramos
        ]
        return np.piecewise(z_arr, condiciones, funciones)

    def radio_cuadrado(self, z: float | np.ndarray) -> np.ndarray:
        """El integrando f(z) = r(z)^2 [cm^2]. Es lo que realmente se integra."""
        return self.radio(z) ** 2

    def volumen_exacto(self) -> float:
        """Volumen analítico total [cm^3]. Este es el VALOR DE REFERENCIA.

        No es una aproximación: es la suma cerrada de los volúmenes de los troncos.
        Contra este número mediremos el error de cada método numérico.
        """
        return sum(t.volumen_exacto() for t in self.tramos)

    def puntos_de_quiebre(self) -> np.ndarray:
        """Alturas donde el perfil es continuo pero NO derivable [cm].

        Son las uniones interiores entre tramos. Importan porque ahí se rompe la
        hipótesis de suavidad de las cotas de error de orden alto.
        """
        return np.array([t.z_fin for t in self.tramos[:-1]])

    def resumen(self) -> pd.DataFrame:
        """Tabla legible de los tramos con su aporte al volumen total."""
        V_total = self.volumen_exacto()
        filas = []
        for i, t in enumerate(self.tramos):
            v = t.volumen_exacto()
            filas.append({
                "#": i,
                "tramo": t.nombre,
                "z_ini [cm]": t.z_ini,
                "z_fin [cm]": t.z_fin,
                "h [cm]": t.altura,
                "r_ini [cm]": t.r_ini,
                "r_fin [cm]": t.r_fin,
                "forma": "cilindro" if math.isclose(t.r_ini, t.r_fin) else "tronco de cono",
                "V tramo [cm³]": v,
                "% del total": 100.0 * v / V_total,
            })
        return pd.DataFrame(filas)


print("Estructuras definidas: Tramo, PerfilEnvase")

### 2.5 Instanciación del envase de prueba

**Qué hace esta celda:** construye el envase concreto con el que trabajaremos: una
botella PET tipo refresco de 600 mL, con proporciones realistas.

**⚙️ Estas son las "proporciones x" — edítalas libremente.** Cambia cualquier número de
`TRAMOS` y todo el notebook se recalcula, incluido el volumen de referencia exacto. Esa
es la ventaja de tener el valor de referencia en forma cerrada: **no hay que remedirlo**.

**Nota importante sobre qué volumen estamos calculando.** El perfil describe la
**superficie exterior**, así que la integral da el **volumen exterior** del envase
(≈ 617 cm³), no su capacidad comercial (600 mL). La diferencia — espesor de pared,
cuello macizo, base cóncava, y el hecho de que nadie llena hasta el borde — es del orden
del 3-8 %.

Esto es coherente con la decisión de validación tomada para el proyecto: contrastaremos
contra el **volumen exterior medido por desplazamiento de agua**, que es la única
cantidad físicamente comparable con lo que se ve en una silueta. Comparar contra la
etiqueta de 600 mL introduciría un sesgo sistemático de ~3 % *antes de empezar a
calcular*, y con un criterio de aprobación del 5 % eso es regalar más de la mitad del
presupuesto de error.

In [ ]:
# =======================================================================
#  ENVASE DE PRUEBA — botella PET tipo refresco, 600 mL nominales
#  Altura total: 21 cm     Diámetro del cuerpo: 6.8 cm
#  ✏️  EDITA ESTOS VALORES para probar otras proporciones.
# =======================================================================
TRAMOS = (
    #      z_ini  z_fin  r_ini  r_fin   nombre
    Tramo(  0.0,   1.0,   2.50,  3.40, "base (pie)"),
    Tramo(  1.0,  15.0,   3.40,  3.40, "cuerpo cilíndrico"),
    Tramo( 15.0,  18.5,   3.40,  1.35, "hombro"),
    Tramo( 18.5,  20.0,   1.35,  1.35, "cuello"),
    Tramo( 20.0,  21.0,   1.35,  1.45, "boca / rosca"),
)

ENVASE = PerfilEnvase(
    tramos=TRAMOS,
    nombre="Botella PET 600 mL",
    capacidad_nominal_ml=600.0,
)

# --- Reporte -----------------------------------------------------------
print("=" * 74)
print(f"ENVASE MODELADO: {ENVASE.nombre}")
print("=" * 74)
print(f"  Intervalo de integración [a, b] : [{ENVASE.a:.3f}, {ENVASE.b:.3f}] cm")
print(f"  Altura total (b - a)            : {ENVASE.altura:.3f} cm")
print(f"  Número de tramos                : {len(ENVASE.tramos)}")
print(f"  Puntos de quiebre (no derivable): {ENVASE.puntos_de_quiebre()} cm")
print(f"  Radio máximo                    : {float(ENVASE.radio(np.linspace(ENVASE.a, ENVASE.b, 2001)).max()):.3f} cm")
print("=" * 74)

display(ENVASE.resumen())

**Cómo leer el resultado:**

- **`[a, b] = [0.000, 21.000]`** — es el intervalo de integración. Toda la sección 5
  trabaja sobre él.
- **Puntos de quiebre `[1.0, 15.0, 18.5, 20.0]`** — las cuatro alturas donde el perfil
  tiene una arista. Guárdalas mentalmente: en la Sección 8 explicarán por qué el orden
  de convergencia observado no coincide con el teórico.
- **La tabla de tramos** muestra el aporte de cada uno. Verás que el **cuerpo
  cilíndrico aporta ~82 %** del volumen. Esto tiene una lectura práctica directa: la
  precisión del resultado final depende abrumadoramente de medir bien **el radio del
  cuerpo**. Un error de 1 mm en el radio del cuello es casi irrelevante; 1 mm en el
  radio del cuerpo son ~18 cm³ (3 % del total). En la fase de cámara, esto dice dónde
  concentrar la calidad de la medición.
- **Columna `forma`** — distingue cilindros de troncos. Sirve de verificación visual de
  que los tramos quedaron como pretendías tras editarlos.

---
## 3. Volumen de referencia exacto

### 3.1 Qué significa "exacto" aquí

**Qué hace esta celda:** calcula el volumen del envase por tres vías independientes y
las contrasta.

1. **Fórmula cerrada del tronco de cono.** Álgebra pura. El único error posible es el
   redondeo de `float64`, del orden de 1e-16 relativo.
2. **`scipy.integrate.quad`** con tolerancia exigida de 1e-13, informándole de los
   puntos de quiebre mediante el argumento `points`. Es cuadratura adaptativa
   (Gauss-Kronrod): subdivide el intervalo donde detecta dificultad.
3. **Suma de Riemann con 2 000 000 de subintervalos.** Deliberadamente rudimentaria.

**Por qué tres métodos y no uno.** Esto es una **verificación cruzada**. La fórmula
cerrada podría tener un error de transcripción; `quad` podría estar mal invocado. Que
tres caminos independientes — uno algebraico, uno adaptativo sofisticado, uno de fuerza
bruta primitivo — converjan al mismo número es evidencia fuerte de que el número es
correcto. Es el mismo principio por el que un laboratorio físico mide una magnitud con
dos instrumentos distintos.

**Por qué `points=` en `quad` no es opcional.** Sin ese argumento, `quad` aplica su
regla adaptativa suponiendo un integrando suave y puede converger lentamente o emitir
una advertencia al tropezar con las aristas. Pasándole los quiebres, **parte el
intervalo en trozos donde el integrando sí es suave** y los resuelve por separado. Es la
misma estrategia que aplicaremos manualmente en la Sección 8 para recuperar el orden
teórico de Simpson.

In [ ]:
# --- Vía 1: fórmula cerrada (la referencia) ----------------------------
V_EXACTO = ENVASE.volumen_exacto()
I_EXACTA = V_EXACTO / math.pi          # el valor exacto de ∫ r² dz  [cm³]

# --- Vía 2: cuadratura adaptativa de alta precisión --------------------
# points= le indica a quad dónde está el integrando quebrado, para que
# subdivida ahí y trate cada trozo como suave.
I_quad, err_quad = integrate.quad(
    lambda zz: float(ENVASE.radio_cuadrado(zz)[0]),
    ENVASE.a, ENVASE.b,
    points=ENVASE.puntos_de_quiebre(),
    epsabs=1e-13, epsrel=1e-13, limit=200,
)
V_quad = math.pi * I_quad

# --- Vía 3: fuerza bruta (suma de Riemann del punto medio) -------------
N_BRUTO = 2_000_000
z_bruto = np.linspace(ENVASE.a, ENVASE.b, N_BRUTO + 1)
z_medios = 0.5 * (z_bruto[:-1] + z_bruto[1:])
dz_bruto = (ENVASE.b - ENVASE.a) / N_BRUTO
V_bruto = math.pi * float(np.sum(ENVASE.radio_cuadrado(z_medios)) * dz_bruto)

# --- Reporte -----------------------------------------------------------
print("=" * 74)
print("VOLUMEN DE REFERENCIA — verificación cruzada por tres vías")
print("=" * 74)
print(f"  1. Fórmula cerrada (tronco de cono) : {V_EXACTO:18.10f} cm³   ← REFERENCIA")
print(f"  2. scipy.integrate.quad (tol 1e-13) : {V_quad:18.10f} cm³")
print(f"  3. Riemann, {N_BRUTO:,} subintervalos : {V_bruto:18.10f} cm³")
print("-" * 74)
print(f"  Discrepancia |1 − 2|                : {abs(V_EXACTO - V_quad):.3e} cm³"
      f"   ({abs(V_EXACTO - V_quad)/V_EXACTO:.3e} relativo)")
print(f"  Discrepancia |1 − 3|                : {abs(V_EXACTO - V_bruto):.3e} cm³"
      f"   ({abs(V_EXACTO - V_bruto)/V_EXACTO:.3e} relativo)")
print("=" * 74)
print(f"  Integral exacta  I = ∫ r² dz        : {I_EXACTA:18.10f} cm³")
print(f"  Volumen exacto   V = π·I            : {V_EXACTO:18.10f} cm³  =  {V_EXACTO:.2f} mL")
if ENVASE.capacidad_nominal_ml:
    exceso = 100.0 * (V_EXACTO - ENVASE.capacidad_nominal_ml) / ENVASE.capacidad_nominal_ml
    print(f"  Capacidad nominal (etiqueta)        : {ENVASE.capacidad_nominal_ml:18.2f} mL")
    print(f"  Exceso del volumen EXTERIOR         : {exceso:+.2f} %"
          f"  ← pared, cuello macizo, base cóncava")
print("=" * 74)

**Cómo leer el resultado:**

- **Las vías 1 y 2 deben coincidir en ~12 dígitos** (discrepancia relativa ~1e-14 o
  menor). Si coinciden, la fórmula cerrada está bien implementada y `quad` bien
  invocado. Este es el número en el que descansa todo el resto del notebook.
- **La vía 3 discrepa más** — alrededor de $2.5\times10^{-11}$ relativo. Y eso es
  **informativo, no un fallo**: la regla del punto medio es de orden $O(h^2)$, así que
  con 2 millones de subintervalos ($h \approx 10^{-5}$) su error esperado es
  $\sim h^2 \approx 10^{-10}$ — coherente con lo observado. Que el error medido siga la
  ley de potencia predicha es, en sí mismo, una validación del código.
- **Ponlo en perspectiva:** esta vía gastó **2 millones** de evaluaciones para llegar a
  ~1e-11 relativo. En la Sección 8 verás reglas de orden alto alcanzar el **suelo de
  redondeo** (~1e-16) con unos pocos cientos de puntos. Cuatro órdenes de magnitud menos
  de trabajo para cinco órdenes de magnitud más de precisión: **el orden del método pesa
  mucho más que la cantidad de puntos**... siempre que el integrando sea suave. Esa
  última cláusula es la que dominará todo el resto del notebook.
- **`V_EXACTO ≈ 617 cm³`** — este es el patrón contra el que se mide todo.
- **El exceso sobre los 600 mL nominales (≈ +3 %)** cuantifica exactamente la trampa
  discutida antes: si validáramos contra la etiqueta, arrancaríamos con 3 % de error
  cuando el criterio de aprobación es 5 %.

---
## 4. Construcción del conjunto de datos numéricos — §7 Etapa 4

### 4.1 El paso conceptual clave de todo el laboratorio

Hasta ahora teníamos $r(z)$ como **función**: podíamos evaluarla en cualquier $z$, con
precisión infinita. Eso es un lujo que en el proyecto real **no vamos a tener**.

De una fotografía no sale una función; sale una **lista finita de pares** $(z_i, r_i)$,
uno por fila de píxeles del contorno. Así que aquí hacemos deliberadamente el mismo
sacrificio: **tiramos la función y nos quedamos solo con la tabla**.

$$r(z) \;\;\xrightarrow{\text{muestreo}}\;\; \{(z_0, r_0), (z_1, r_1), \dots, (z_n, r_n)\}$$

A partir de este punto, los métodos de integración verán **exactamente el mismo tipo de
dato que verán en la fase de cámara**. Eso es lo que hace que las conclusiones de este
notebook sean transferibles: no estamos midiendo el error de integrar una función bonita,
estamos midiendo el error de integrar *una tabla*.

### 4.2 Malla uniforme y el diferencial Δz

Muestreamos con **espaciamiento constante**, como pide la §5 del enunciado:

$$\Delta z = \frac{b-a}{n}, \qquad z_i = a + i\,\Delta z, \quad i = 0,1,\dots,n$$

Dos advertencias sobre el conteo, porque es fuente constante de errores de programación:

- $n$ = número de **subintervalos** (los huecos entre puntos).
- $n+1$ = número de **puntos** (nodos).

Simpson pide "$n$ par" refiriéndose a **subintervalos**, es decir, un número *impar* de
puntos. Confundirlos es el error clásico. En el código, `n_subintervalos` y
`n_puntos` son campos separados y explícitos precisamente para que nadie tenga que
recordar cuál es cuál.

### 4.3 Por qué elegimos n = 60 como caso base

Los cuatro métodos imponen restricciones distintas sobre $n$:

| Método | Requiere |
|---|---|
| Trapecio | cualquier $n \ge 1$ |
| Simpson 1/3 | $n$ **par** (múltiplo de 2) |
| Simpson 3/8 | $n$ múltiplo de **3** |
| Boole | $n$ múltiplo de **4** |

Para que **los cuatro operen sobre la misma tabla** — lo cual exige literalmente la §7
Etapa 5 del PDF: *"utilizar exactamente el mismo perfil de entrada"* — necesitamos que
$n$ sea múltiplo del **mínimo común múltiplo**:

$$\mathrm{mcm}(2, 3, 4) = 12$$

$n = 60 = 5 \times 12$ cumple las cuatro condiciones, y da $\Delta z = 0.35$ cm, una
resolución comparable a la que dará una foto de celular. Más adelante, la Sección 8
barrerá muchos valores de $n$; aquí fijamos uno para poder inspeccionar la tabla a
mano.

In [ ]:
def mcm(a: int, b: int) -> int:
    """Mínimo común múltiplo de dos enteros."""
    return a * b // math.gcd(a, b)


# mcm(2,3,4) = 12 → todo múltiplo de 12 sirve para los cuatro métodos a la vez.
N_COMPATIBLE = mcm(mcm(2, 3), 4)


def n_compatible_mas_cercano(n_deseado: int) -> int:
    """Devuelve el múltiplo de 12 más cercano a n_deseado (mínimo 12).

    Sirve para responder al usuario cuando pide un n que no sirve para todos los
    métodos: en lugar de solo rechazarlo, le proponemos el valor válido más próximo.
    Rechazar sin sugerir es un mal mensaje de error.
    """
    if n_deseado <= N_COMPATIBLE:
        return N_COMPATIBLE
    abajo = (n_deseado // N_COMPATIBLE) * N_COMPATIBLE
    arriba = abajo + N_COMPATIBLE
    return abajo if (n_deseado - abajo) <= (arriba - n_deseado) else arriba


def muestrear_perfil(perfil: PerfilEnvase, n: int) -> tuple[np.ndarray, np.ndarray]:
    """Convierte el perfil continuo en una tabla discreta (z_i, r_i).

    Este es el punto donde renunciamos a la función y nos quedamos con datos,
    replicando lo que ocurrirá al extraer el contorno de una fotografía.

    Parámetros
    ----------
    perfil : PerfilEnvase   El envase a muestrear.
    n      : int            Número de SUBINTERVALOS (se generan n+1 puntos).

    Devuelve
    --------
    (z, r) : dos arrays de longitud n+1, en cm.
    """
    if n < 1:
        raise ValueError(f"n debe ser >= 1, se recibió {n}.")
    z = np.linspace(perfil.a, perfil.b, n + 1)   # linspace incluye ambos extremos
    r = perfil.radio(z)
    return z, r


def tabla_datos(z: np.ndarray, r: np.ndarray) -> pd.DataFrame:
    """Construye la tabla que exige la §7 Etapa 4 del enunciado.

    Columnas: i | z_i | r_i | r_i² | f(z_i) = π·r_i²

    La columna r² es el integrando real; la columna π·r² es el área del disco a esa
    altura. Ambas se muestran para que el cálculo sea verificable a mano, que es
    justamente lo que pide el enunciado.
    """
    return pd.DataFrame({
        "i": np.arange(len(z)),
        "z_i [cm]": z,
        "r_i [cm]": r,
        "r_i² [cm²]": r ** 2,
        "f(z_i) = π·r_i² [cm²]": math.pi * r ** 2,
    })


# =======================================================================
#  CASO BASE
# =======================================================================
N_BASE = 60                                   # ✏️ editable (múltiplo de 12)
Z_BASE, R_BASE = muestrear_perfil(ENVASE, N_BASE)
DZ_BASE = (ENVASE.b - ENVASE.a) / N_BASE
TABLA_BASE = tabla_datos(Z_BASE, R_BASE)

print("=" * 74)
print("CONJUNTO DE DATOS NUMÉRICOS  (§7 Etapa 4)")
print("=" * 74)
print(f"  Intervalo [a, b]          : [{ENVASE.a:.4f}, {ENVASE.b:.4f}] cm")
print(f"  n (subintervalos)         : {N_BASE}")
print(f"  Puntos (nodos) = n + 1    : {N_BASE + 1}")
print(f"  Δz = (b − a)/n            : {DZ_BASE:.6f} cm   ← EL DIFERENCIAL")
print(f"  Malla uniforme            : sí")
print("-" * 74)
print(f"  Compatibilidad de n = {N_BASE}:")
for nombre, req in (("Trapecio", 1), ("Simpson 1/3", 2), ("Simpson 3/8", 3), ("Boole", 4)):
    ok = "✓" if N_BASE % req == 0 else "✗"
    print(f"     {ok} {nombre:<14} requiere n múltiplo de {req}"
          f"   →  {N_BASE} % {req} = {N_BASE % req}")
print("=" * 74)

print("\nPrimeras 8 filas:")
display(TABLA_BASE.head(8))
print("Últimas 5 filas:")
display(TABLA_BASE.tail(5))

**Cómo leer el resultado:**

- **`Δz = 0.35 cm`** es *el diferencial de discretización* que la §5 del PDF exige
  hacer explícito. Es la distancia física entre dos filas consecutivas de la tabla. En
  la fase de cámara, Δz será literalmente **la altura de un píxel expresada en
  centímetros**.
- **61 puntos para 60 subintervalos.** Compruébalo en la tabla: el índice `i` llega
  hasta 60, así que hay 61 filas. Esta distinción es la que rige las condiciones de
  Simpson y Boole.
- **Las cuatro marcas ✓** confirman que $n=60$ sirve para los cuatro métodos, de modo
  que los compararemos sobre datos idénticos — sin esto, cualquier diferencia entre
  métodos sería incomparable, porque estarían integrando tablas distintas.
- **La columna `r_i`** en las primeras filas crece de 2.50 a 3.40: es el pie del envase
  ensanchándose. En las últimas filas ronda 1.35-1.45: el cuello y la boca.
- **La columna `f(z_i) = π·r_i²`** es el área del disco a esa altura, en cm². El volumen
  es el área bajo *esta* curva. Nota que $\pi r^2$ amplifica: pasar de $r$=3.40 a
  $r$=1.35 (una caída del 60 %) hace caer $f$ de 36.3 a 5.7 cm² (una caída del 84 %).
  Esa es la no linealidad del cuadrado, visualmente.

---
## 5. Los cuatro métodos de integración numérica — §6

### 5.1 Infraestructura común

**Qué hace esta celda:** define dos piezas de soporte que usarán los cuatro métodos.

**`ResultadoIntegracion`** — un contenedor inmutable con *todo* lo que la §7 Etapa 5 del
enunciado obliga a reportar:

> *número de subintervalos o puntos utilizados; intervalo [a,b]; Δz o la discretización
> equivalente; orden o número de puntos de cuadratura cuando aplique; valor de la
> integral estimada; volumen estimado; tiempo de cálculo.*

**Por qué un dataclass y no imprimir dentro de cada función.** Separar el **cálculo** de
la **presentación** es una regla básica de diseño, y aquí tiene un beneficio concreto e
inmediato: en la Sección 8 ejecutaremos los métodos ~200 veces para el estudio de
convergencia. Si cada uno imprimiera, tendríamos 200 bloques de texto inútiles. Al
devolver objetos, los acumulamos en una lista, los volcamos a un DataFrame y los
graficamos. **Una función que calcula e imprime a la vez no se puede reutilizar.**

**`CondicionNoSatisfecha`** — una excepción propia. El PDF (§8) pide explícitamente
*"mensajes claros cuando los datos no cumplen las condiciones requeridas por un
método"*. Una excepción específica permite a la interfaz distinguir *"este método no
aplica a estos datos"* (recuperable: se avisa y se sigue con los demás) de *"el código
tiene un bug"* (no recuperable). Sus mensajes explican **qué falló, por qué, y cuál es
el valor válido más cercano** — los tres ingredientes de un buen mensaje de error.

In [ ]:
class CondicionNoSatisfecha(Exception):
    """Los datos no cumplen los requisitos del método de integración solicitado.

    Es un error ESPERADO y recuperable, no un bug: significa que este método
    concreto no es aplicable a esta tabla concreta. La interfaz debe capturarlo,
    informar al usuario y continuar con los demás métodos.
    """
    pass


@dataclass(frozen=True)
class ResultadoIntegracion:
    """Resultado completo de aplicar un método de cuadratura.

    Contiene todos los campos que la §7 Etapa 5 del enunciado exige reportar, de
    modo que el objeto por sí solo es un registro auditable del cálculo.
    """
    metodo: str                 # nombre legible
    familia: str                # "Newton-Cotes cerrada" | "Cuadratura gaussiana"
    n_subintervalos: int        # n
    n_evaluaciones: int         # cuántas veces se evaluó el integrando
    a: float                    # límite inferior [cm]
    b: float                    # límite superior [cm]
    delta_z: float | None       # Δz [cm]; None si la malla no es uniforme (Gauss)
    orden_error: str            # orden teórico del error global
    grado_exactitud: int        # máximo grado polinómico integrado exactamente
    integral_r2: float          # I = ∫ r² dz  [cm³]
    volumen_cm3: float          # V = π·I      [cm³]
    tiempo_ms: float            # tiempo de cálculo [ms]
    notas: str = ""             # observaciones (p. ej. composición de reglas)

    @property
    def volumen_ml(self) -> float:
        """Volumen en mL. 1 cm³ = 1 mL exactamente, por definición del litro."""
        return self.volumen_cm3

    def error_vs(self, v_referencia: float) -> tuple[float, float]:
        """Error absoluto [cm³] y error relativo porcentual frente a una referencia.

        Son las dos métricas que pide la §7 Etapa 6 del enunciado.
        """
        abs_err = abs(self.volumen_cm3 - v_referencia)
        rel_err = 100.0 * abs_err / abs(v_referencia)
        return abs_err, rel_err

    def informe(self, v_referencia: float | None = None) -> str:
        """Bloque de texto con todos los parámetros del método. Para el informe."""
        dz = "no uniforme" if self.delta_z is None else f"{self.delta_z:.6f} cm"
        lineas = [
            f"  Método                  : {self.metodo}",
            f"  Familia                 : {self.familia}",
            f"  Intervalo [a, b]        : [{self.a:.4f}, {self.b:.4f}] cm",
            f"  n (subintervalos)       : {self.n_subintervalos}",
            f"  Evaluaciones de r(z)    : {self.n_evaluaciones}",
            f"  Δz                      : {dz}",
            f"  Orden del error global  : {self.orden_error}",
            f"  Grado de exactitud      : {self.grado_exactitud} "
            f"(integra exactamente polinomios de grado ≤ {self.grado_exactitud})",
            f"  Integral I = ∫r²dz      : {self.integral_r2:.10f} cm³",
            f"  Volumen  V = π·I        : {self.volumen_cm3:.10f} cm³  = {self.volumen_ml:.4f} mL",
            f"  Tiempo de cálculo       : {self.tiempo_ms:.4f} ms",
        ]
        if v_referencia is not None:
            e_abs, e_rel = self.error_vs(v_referencia)
            lineas += [
                f"  Error absoluto          : {e_abs:.3e} cm³",
                f"  Error relativo          : {e_rel:.3e} %",
            ]
        if self.notas:
            lineas.append(f"  Notas                   : {self.notas}")
        return "\n".join(lineas)


def _cronometrar(funcion: Callable[[], float], repeticiones: int = 7) -> tuple[float, float]:
    """Ejecuta `funcion` varias veces y devuelve (valor, tiempo_mínimo_en_ms).

    Se toma el MÍNIMO, no el promedio. Razón: el tiempo medido es el tiempo real
    de cálculo MÁS el ruido del sistema (el planificador del SO interrumpe, otro
    proceso usa la CPU, entra el recolector de basura). Ese ruido es siempre
    positivo — nunca hace que algo vaya más rápido de lo que puede. Por tanto el
    mínimo de varias corridas es la mejor estimación del coste real.
    Es la misma razón por la que el módulo `timeit` de Python reporta el mínimo.
    """
    mejor = float("inf")
    valor = 0.0
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        valor = funcion()
        t1 = time.perf_counter()
        mejor = min(mejor, (t1 - t0) * 1000.0)
    return valor, mejor


def _validar_malla(z: np.ndarray, r: np.ndarray, tol: float = 1e-9) -> tuple[int, float]:
    """Verifica que (z, r) sea una malla uniforme válida. Devuelve (n, Δz).

    Las fórmulas de Newton-Cotes compuestas que implementamos ABAJO suponen
    espaciamiento constante. Aplicarlas a datos no uniformes daría un número
    plausible pero incorrecto — el peor tipo de fallo. Por eso se verifica aquí,
    una sola vez, en lugar de confiar en que quien llame lo haya hecho bien.
    """
    z = np.asarray(z, dtype=float)
    r = np.asarray(r, dtype=float)
    if z.ndim != 1 or r.ndim != 1:
        raise CondicionNoSatisfecha("z y r deben ser arrays unidimensionales.")
    if len(z) != len(r):
        raise CondicionNoSatisfecha(
            f"z y r deben tener la misma longitud (z: {len(z)}, r: {len(r)})."
        )
    if len(z) < 2:
        raise CondicionNoSatisfecha(
            f"Se requieren al menos 2 puntos para integrar; se recibieron {len(z)}."
        )
    if not np.all(np.diff(z) > 0):
        raise CondicionNoSatisfecha(
            "Las alturas z deben ser estrictamente crecientes. "
            "Ordena los datos antes de integrar."
        )
    difs = np.diff(z)
    dz = float(difs[0])
    desviacion = float(np.max(np.abs(difs - dz)))
    if desviacion > tol * max(1.0, abs(dz)):
        raise CondicionNoSatisfecha(
            f"La malla NO es uniforme: el espaciamiento varía hasta {desviacion:.2e} cm "
            f"(Δz nominal = {dz:.6f} cm). Las fórmulas de Newton-Cotes compuestas "
            f"requieren espaciamiento constante. Solución: remuestrea el perfil a una "
            f"malla uniforme por interpolación (spline) antes de integrar, con n "
            f"múltiplo de 12 para que los cuatro métodos sean aplicables."
        )
    return len(z) - 1, dz


print("Infraestructura lista: ResultadoIntegracion, CondicionNoSatisfecha,")
print("                       _cronometrar, _validar_malla")

**Cómo leer el resultado:** solo confirma que las definiciones se cargaron. La
utilidad de estas piezas se verá en las celdas siguientes.

Dos detalles de diseño que conviene notar porque son decisiones deliberadas:

- **`_cronometrar` devuelve el mínimo de 7 corridas, no el promedio.** El tiempo medido
  es el coste real *más* ruido del sistema operativo, y ese ruido solo puede ser
  positivo. El mínimo es por tanto el estimador menos contaminado del coste verdadero.
  Es exactamente el criterio del módulo `timeit` de la biblioteca estándar.
- **`_validar_malla` rechaza mallas no uniformes.** Podríamos haber dejado que las
  fórmulas se aplicaran igual: no habrían fallado, habrían devuelto un número. Un número
  **incorrecto**, sin ninguna señal de alarma. Fallar ruidosamente ante una precondición
  violada es siempre preferible a devolver silenciosamente basura verosímil —
  especialmente en un sistema de medición, donde nadie va a sospechar de un volumen que
  "se ve razonable".

### 5.2 Regla del trapecio

#### La idea

Entre dos nodos consecutivos, sustituimos el integrando por la **recta** que une sus
valores. El área bajo esa recta es la de un trapecio:

$$\int_{z_i}^{z_{i+1}} f(z)\,dz \;\approx\; \frac{\Delta z}{2}\,\bigl[f(z_i) + f(z_{i+1})\bigr]$$

Sumando sobre los $n$ subintervalos, los nodos interiores aparecen **dos veces** (como
extremo derecho de un trapecio y como extremo izquierdo del siguiente), y los extremos
solo una:

$$\boxed{\;I \;\approx\; \frac{\Delta z}{2}\left[f_0 + 2\sum_{i=1}^{n-1} f_i + f_n\right]\;}$$

De ahí el patrón de pesos $[1, 2, 2, \dots, 2, 1]$.

#### Error

$$E \;=\; -\frac{(b-a)\,\Delta z^{2}}{12}\,f''(\xi), \qquad \xi \in (a,b)$$

Es **$O(\Delta z^2)$**: al **duplicar** el número de puntos, el error cae a **la cuarta
parte**. Lo verificaremos numéricamente en la Sección 8.

#### Grado de exactitud: 1

Integra exactamente rectas ($f$ de grado ≤ 1), y nada más. Nuestro integrando es
$f = r^2$, que en el cuerpo cilíndrico es **constante** (grado 0 — exacto ahí) pero en
la base, el hombro y la boca es **cuadrático** (porque $r$ es lineal y lo elevamos al
cuadrado). El trapecio falla en esas tres zonas, y ahí se concentra todo su error.

#### El signo del error: una predicción que vamos a ver fallar

El razonamiento habitual dice: como $f'' > 0$ en los tramos cónicos ($f = r^2$ con $r$
lineal es una parábola convexa), la cuerda queda **por encima** de la curva y el trapecio
**sobreestima**.

Ese razonamiento es correcto **dentro de cada tramo**, y el notebook lo verificará. Pero
el resultado global va a salir **al revés**: el trapecio **subestima**. La razón está en
los codos.

En un punto de quiebre, $f'$ salta de golpe. Eso significa que $f''$ contiene un
**impulso** (una delta de Dirac) justo ahí. En el hombro y en el pie, la pendiente cae
bruscamente — de creciente a constante en $z=1$, de constante a decreciente en $z=15$ —
así que esos impulsos son **negativos**. Y un impulso negativo de $f''$ hace que el
trapecio **subestime** en ese subintervalo.

Con malla ciega, los codos caen en mitad de un subintervalo y esas contribuciones
negativas **superan** a la sobreestimación convexa repartida por los tramos suaves. El
signo global se invierte.

> **La lección:** una cota de error del tipo $E = -\frac{(b-a)h^2}{12}f''(\xi)$ presupone
> que $f''$ **existe y es acotada**. Cuando el integrando tiene esquinas, $f''$ no es una
> función acotada, y la fórmula deja de predecir incluso el *signo*. Lo comprobaremos en
> los dos sentidos: globalmente sale negativo, y tramo a tramo sale positivo.

In [ ]:
def regla_trapecio(z: np.ndarray, r: np.ndarray) -> ResultadoIntegracion:
    """Regla del trapecio compuesta sobre la tabla (z, r).

    Integra f = r² con pesos [1, 2, 2, ..., 2, 1] · Δz/2.

    Condiciones: malla uniforme, n >= 1. No hay restricción de paridad, lo que la
    convierte en el método de respaldo universal cuando los demás no aplican.
    """
    n, dz = _validar_malla(z, r)
    f = np.asarray(r, dtype=float) ** 2          # el integrando: f(z) = r(z)²

    def _calcular() -> float:
        # Pesos explícitos en lugar de np.trapz: el objetivo del laboratorio es que
        # la fórmula sea visible y auditable, no que sea corta.
        pesos = np.full(n + 1, 2.0)
        pesos[0] = pesos[-1] = 1.0
        return float(dz / 2.0 * np.dot(pesos, f))

    integral, ms = _cronometrar(_calcular)
    return ResultadoIntegracion(
        metodo="Trapecio compuesto",
        familia="Newton-Cotes cerrada (grado 1)",
        n_subintervalos=n,
        n_evaluaciones=n + 1,
        a=float(z[0]), b=float(z[-1]),
        delta_z=dz,
        orden_error="O(Δz²)",
        grado_exactitud=1,
        integral_r2=integral,
        volumen_cm3=math.pi * integral,
        tiempo_ms=ms,
        notas="Sin restricción sobre n. Sobreestima si el integrando es convexo.",
    )


res_trap = regla_trapecio(Z_BASE, R_BASE)
print("=" * 74)
print("MÉTODO 1 — REGLA DEL TRAPECIO")
print("=" * 74)
print(res_trap.informe(V_EXACTO))
print("=" * 74)
_signo = "SOBREESTIMA" if res_trap.volumen_cm3 > V_EXACTO else "SUBESTIMA"
print(f"  Diferencia con signo: {res_trap.volumen_cm3 - V_EXACTO:+.6f} cm³  → {_signo}")

**Cómo leer el resultado:**

- **Error relativo ≈ 1.0 × 10⁻² %** — unos 0.063 cm³ sobre 617. Es **0.2 % del margen
  del 5 %** que exige el reto. **El trapecio, el método más humilde de todos, ya basta
  sobradamente para este problema.** Conviene interiorizarlo desde ya: el cuello de
  botella de este proyecto no está aquí.
- **`Diferencia con signo` NEGATIVA → SUBESTIMA.** Es la predicción "fallida" de la
  sección anterior, y no es casualidad ni ruido: si reejecutas con $n = 12, 24, 120, 240,
  480$ el signo sigue siendo negativo en todos los casos. Los impulsos negativos de
  $f''$ en los codos dominan sistemáticamente sobre la convexidad de los tramos.
  **La cota de error clásica no predice ni siquiera el signo cuando el integrando tiene
  esquinas.** Lo verificaremos en la Sección 8 integrando tramo a tramo: ahí el signo
  sí sale positivo, como manda la teoría.
- **`Evaluaciones = 61`** — el trapecio usa todos los nodos y ninguno extra. Es el
  método más barato en evaluaciones por unidad de dato.
- **`Grado de exactitud = 1`** — en el cuerpo cilíndrico $f = r^2$ es constante (grado 0)
  y el trapecio es **exacto** ahí. Como ese tramo aporta el 82 % del volumen, el trapecio
  acierta de entrada la mayor parte del resultado. Todo su error proviene de los tramos
  cónicos y de los codos.

### 5.3 Regla de Simpson 1/3

#### La idea

En lugar de una recta por cada subintervalo, ajustamos una **parábola por cada *par* de
subintervalos**. Tres puntos definen una parábola; integrándola analíticamente sobre los
dos subintervalos que abarca:

$$\int_{z_i}^{z_{i+2}} f\,dz \;\approx\; \frac{\Delta z}{3}\bigl[f_i + 4f_{i+1} + f_{i+2}\bigr]$$

Compuesta sobre todo el intervalo:

$$\boxed{\;I \;\approx\; \frac{\Delta z}{3}\left[f_0 + 4\!\!\sum_{i\ \text{impar}}\!\! f_i \;+\; 2\!\!\sum_{\substack{i\ \text{par}\\ 0<i<n}}\!\! f_i \;+\; f_n\right]\;}$$

Patrón de pesos: $[1, 4, 2, 4, 2, \dots, 4, 1]$. Los índices **impares** son centros de
parábola (peso 4); los **pares interiores** son costuras compartidas entre dos parábolas
(peso 2 = 1+1).

#### La condición: n debe ser par

Cada parábola consume exactamente 2 subintervalos. Si $n$ fuese impar sobraría uno
suelto, sin pareja. **Esta es una de las condiciones cuyo incumplimiento el enunciado
(§8) obliga a reportar con un mensaje claro** — lo probaremos deliberadamente en la
Sección 6.

#### Error — y la sorpresa que lo hace tan bueno

$$E \;=\; -\frac{(b-a)\,\Delta z^{4}}{180}\,f^{(4)}(\xi)$$

**$O(\Delta z^4)$**: al duplicar los puntos, el error cae a **1/16**.

Lo notable es el **grado de exactitud: 3, no 2**. Construimos la fórmula con parábolas
(grado 2), pero resulta exacta también para **cúbicas**. La razón es una cancelación por
simetría: el error de la parábola sobre la mitad izquierda del intervalo es igual y de
signo opuesto al de la mitad derecha, y se anulan al sumar. Es un regalo geométrico: se
obtiene un grado gratis. Por eso Simpson 1/3 tiene una relación
**precisión/coste extraordinaria** y es la regla más usada en la práctica.

#### Implicación directa para nuestro envase — y una predicción incómoda

$f = r^2$ es a lo sumo **cuadrático por tramos** (grado 2 ≤ 3). Por tanto, **dentro de
cada tramo, Simpson es exacto salvo redondeo**. Todo el error que observemos vendrá
únicamente de las parábolas que queden **a caballo sobre un punto de quiebre**, donde el
integrando cambia de fórmula a mitad de camino.

Aquí conviene anticipar algo que sorprende: como cada parábola de Simpson abarca **dos**
subintervalos, mientras que cada trapecio abarca **uno**, las parábolas **cruzan codos
con el doble de frecuencia** que los trapecios. Y cuando una parábola se ajusta a tres
puntos que están sobre dos curvas distintas, el resultado no se parece a ninguna de las
dos.

Es perfectamente posible, entonces, que **Simpson salga peor que el trapecio** sobre este
envase. Suena a contradicción con todo lo que promete su cota $O(\Delta z^4)$, pero esa
cota exige $f^{(4)}$ acotada — y en un codo ni siquiera existe $f''$. Vamos a verlo.

In [ ]:
def regla_simpson_13(z: np.ndarray, r: np.ndarray) -> ResultadoIntegracion:
    """Regla de Simpson 1/3 compuesta.

    Pesos [1,4,2,4,...,4,1] · Δz/3.
    Condición: n PAR (cada parábola abarca 2 subintervalos).
    """
    n, dz = _validar_malla(z, r)
    if n % 2 != 0:
        raise CondicionNoSatisfecha(
            f"Simpson 1/3 requiere un número PAR de subintervalos y se recibió n = {n}.\n"
            f"  Motivo: cada parábola se ajusta sobre 2 subintervalos consecutivos; "
            f"con n impar queda uno sin pareja.\n"
            f"  Soluciones: usar n = {n - 1} o n = {n + 1} (remuestreando los datos), "
            f"o aplicar Simpson 3/8 al último tramo."
        )
    f = np.asarray(r, dtype=float) ** 2

    def _calcular() -> float:
        pesos = np.ones(n + 1)
        pesos[1:-1:2] = 4.0     # índices impares → centros de parábola
        pesos[2:-1:2] = 2.0     # índices pares interiores → costuras
        return float(dz / 3.0 * np.dot(pesos, f))

    integral, ms = _cronometrar(_calcular)
    return ResultadoIntegracion(
        metodo="Simpson 1/3 compuesto",
        familia="Newton-Cotes cerrada (grado 2)",
        n_subintervalos=n,
        n_evaluaciones=n + 1,
        a=float(z[0]), b=float(z[-1]),
        delta_z=dz,
        orden_error="O(Δz⁴)",
        grado_exactitud=3,
        integral_r2=integral,
        volumen_cm3=math.pi * integral,
        tiempo_ms=ms,
        notas="Requiere n par. Exacto hasta cúbicas por cancelación simétrica del error.",
    )


res_s13 = regla_simpson_13(Z_BASE, R_BASE)
print("=" * 74)
print("MÉTODO 2 — REGLA DE SIMPSON 1/3")
print("=" * 74)
print(res_s13.informe(V_EXACTO))
print("=" * 74)
_e_trap = res_trap.error_vs(V_EXACTO)[0]
_e_s13 = res_s13.error_vs(V_EXACTO)[0]
_cociente = _e_s13 / max(_e_trap, 1e-300)
print(f"  Error del trapecio    : {_e_trap:.6e} cm³   (orden teórico O(Δz²))")
print(f"  Error de Simpson 1/3  : {_e_s13:.6e} cm³   (orden teórico O(Δz⁴))")
if _cociente > 1:
    print(f"\n  ⚠ Simpson 1/3 es {_cociente:.2f}× PEOR que el trapecio, pese a ser de")
    print(f"    orden teórico muy superior. Causa: los puntos de quiebre del envase.")
    print(f"    Cada parábola abarca 2 subintervalos y cruza codos el doble de veces")
    print(f"    que un trapecio. Ver Sección 8 para la demostración cuantitativa.")
else:
    print(f"\n  Simpson 1/3 es {1/_cociente:.2f}× más preciso que el trapecio.")

**Cómo leer el resultado — aquí está la primera sorpresa del notebook.**

- **Simpson 1/3 sale ≈ 4.2 veces PEOR que el trapecio** (≈ 0.263 cm³ frente a
  ≈ 0.063 cm³), con **exactamente los mismos 61 puntos**. Un método de orden teórico
  $O(\Delta z^4)$ derrotado por uno de orden $O(\Delta z^2)$.
- **Esto no es un error de implementación.** La verificación independiente de las
  fórmulas confirma que Simpson integra exactamente polinomios hasta grado 3 y falla en
  grado 4, justo como debe. El código está bien; **lo que no se cumple son las hipótesis
  de la cota de error**.
- **El mecanismo, en una frase:** cada parábola de Simpson abarca 2 subintervalos, así
  que cruza los codos del envase con el doble de frecuencia que un trapecio, y ajustar
  una parábola a tres puntos repartidos entre dos curvas distintas produce un error
  mayor que el de dos segmentos rectos.
- **El error no es cero pese a ser $f$ cuadrático por tramos**, exactamente por lo mismo:
  con $n = 60$ los quiebres ($z = 1.0,\ 15.0,\ 18.5,\ 20.0$) caen *dentro* de parábolas
  en vez de sobre nodos. En la Sección 8 alinearemos la malla y el error de Simpson caerá
  al **suelo de redondeo** — confirmando que la teoría es correcta y que el problema era
  dónde pusimos los nodos.
- **`Grado de exactitud = 3`** — el grado "regalado" por la cancelación simétrica.
- **El tiempo es prácticamente idéntico** al del trapecio: ambos son un producto escalar
  de 61 términos. El coste no distingue a los métodos; la estructura de los datos, sí.

> **Conclusión provisional, que la Sección 8 confirmará:** sobre datos con esquinas —y
> una foto de un envase real tiene esquinas *y* ruido— **el orden teórico de un método
> no predice su rendimiento**. Elegir la malla bien importa más que elegir la fórmula
> bien.

### 5.4 Regla de Simpson 3/8 — Newton-Cotes cerrada de grado 3

Esta es la fórmula que el PDF (§6) pide explícitamente: *"una fórmula cerrada de
Newton-Cotes, como 3/8 o una de orden superior"*.

#### La idea

Subimos un grado: ajustamos una **cúbica** por cada **tres** subintervalos (4 puntos
determinan una cúbica). Integrándola:

$$\int_{z_i}^{z_{i+3}} f\,dz \;\approx\; \frac{3\,\Delta z}{8}\bigl[f_i + 3f_{i+1} + 3f_{i+2} + f_{i+3}\bigr]$$

El nombre viene del coeficiente $3/8$. Patrón compuesto:
$[1, 3, 3, 2, 3, 3, 2, \dots, 3, 3, 1]$ — los pesos 2 son las costuras entre cúbicas
consecutivas (1+1).

#### Condición: n múltiplo de 3

#### La lección contraintuitiva: más grado NO significa más orden

$$E \;=\; -\frac{(b-a)\,\Delta z^{4}}{80}\,f^{(4)}(\xi)$$

Compárala con la de Simpson 1/3, que tiene $180$ en el denominador:

| | Grado del polinomio | Grado de exactitud | Orden del error | Constante |
|---|---|---|---|---|
| Simpson 1/3 | 2 (parábola) | 3 | $O(\Delta z^4)$ | $1/180$ |
| Simpson 3/8 | 3 (cúbica) | 3 | $O(\Delta z^4)$ | $1/80$ |

**Simpson 3/8 usa polinomios de mayor grado, consume más puntos por segmento, y tiene el
mismo orden y una constante de error 2.25 veces PEOR.**

¿Por qué? Porque la cancelación simétrica que le regalaba un grado extra a Simpson 1/3
**solo ocurre en las reglas de Newton-Cotes de grado par**. Simpson 1/3 (grado 2, par)
se beneficia; Simpson 3/8 (grado 3, impar) no. Es un patrón general:

> En Newton-Cotes cerradas, las reglas de **grado par** siempre ganan un grado de
> exactitud extra. Por eso 1/3 domina a 3/8, y por eso Boole (grado 4, par) es el
> siguiente salto real de orden.

#### Entonces, ¿para qué sirve 3/8?

Su utilidad real es de **compatibilidad**: cuando tienes un $n$ impar, Simpson 1/3 no
aplica. La solución estándar es aplicar 3/8 a los **tres últimos** subintervalos y 1/3 al
resto. Dado que $n$ impar $-3$ = par, el reparto siempre funciona. Es la herramienta que
te salva cuando el número de puntos te viene impuesto — que es exactamente lo que pasa
cuando los datos vienen de una foto y no puedes elegir cuántas filas de píxeles tiene.

Nuestra implementación hace justo eso: **compone automáticamente** ambas reglas si $n$
no es múltiplo de 3, en vez de rendirse.

In [ ]:
def regla_simpson_38(z: np.ndarray, r: np.ndarray,
                     permitir_composicion: bool = True) -> ResultadoIntegracion:
    """Regla de Simpson 3/8 compuesta (Newton-Cotes cerrada de grado 3).

    Pesos [1,3,3,2,3,3,2,...,3,3,1] · 3Δz/8.
    Condición nativa: n múltiplo de 3.

    Si n NO es múltiplo de 3 y permitir_composicion=True, se aplica 3/8 al mayor
    bloque múltiplo de 3 posible y Simpson 1/3 al resto (que queda par por
    construcción). Es la práctica estándar y evita rechazar datos utilizables.
    """
    n, dz = _validar_malla(z, r)
    f = np.asarray(r, dtype=float) ** 2

    resto = n % 3
    if resto != 0 and not permitir_composicion:
        raise CondicionNoSatisfecha(
            f"Simpson 3/8 requiere n múltiplo de 3 y se recibió n = {n} "
            f"(n % 3 = {resto}).\n"
            f"  Motivo: cada cúbica se ajusta sobre 3 subintervalos consecutivos.\n"
            f"  Soluciones: usar n = {n - resto} o n = {n + (3 - resto)}, o activar "
            f"permitir_composicion=True para combinar 3/8 con Simpson 1/3."
        )

    def _pesos_38(m: int) -> np.ndarray:
        """Pesos de 3/8 compuesta sobre m subintervalos (m múltiplo de 3)."""
        w = np.zeros(m + 1)
        for i in range(0, m, 3):        # bloques de 3 subintervalos
            w[i]     += 1.0
            w[i + 1] += 3.0
            w[i + 2] += 3.0
            w[i + 3] += 1.0
        return w * (3.0 / 8.0)

    def _pesos_13(m: int) -> np.ndarray:
        """Pesos de Simpson 1/3 compuesta sobre m subintervalos (m par)."""
        w = np.ones(m + 1)
        w[1:-1:2] = 4.0
        w[2:-1:2] = 2.0
        return w * (1.0 / 3.0)

    if resto == 0:
        nota = "n múltiplo de 3: regla 3/8 pura."
        def _calcular() -> float:
            return float(dz * np.dot(_pesos_38(n), f))
    else:
        # El bloque de 3/8 debe ser múltiplo de 3; el sobrante va a Simpson 1/3
        # y por tanto debe quedar par.
        n_38 = (n // 3) * 3
        n_13 = n - n_38
        if n_13 % 2 != 0:                 # el sobrante debe ser par para 1/3
            n_38 -= 3
            n_13 = n - n_38
        nota = (f"n = {n} no es múltiplo de 3: composición automática → "
                f"3/8 sobre los primeros {n_38} subintervalos + "
                f"Simpson 1/3 sobre los últimos {n_13}.")
        def _calcular() -> float:
            total = 0.0
            if n_38 > 0:
                total += float(dz * np.dot(_pesos_38(n_38), f[: n_38 + 1]))
            if n_13 > 0:
                total += float(dz * np.dot(_pesos_13(n_13), f[n_38:]))
            return total

    integral, ms = _cronometrar(_calcular)
    return ResultadoIntegracion(
        metodo="Simpson 3/8 compuesto",
        familia="Newton-Cotes cerrada (grado 3)",
        n_subintervalos=n,
        n_evaluaciones=n + 1,
        a=float(z[0]), b=float(z[-1]),
        delta_z=dz,
        orden_error="O(Δz⁴)",
        grado_exactitud=3,
        integral_r2=integral,
        volumen_cm3=math.pi * integral,
        tiempo_ms=ms,
        notas=nota,
    )


res_s38 = regla_simpson_38(Z_BASE, R_BASE)
print("=" * 74)
print("MÉTODO 3 — REGLA DE SIMPSON 3/8  (Newton-Cotes cerrada)")
print("=" * 74)
print(res_s38.informe(V_EXACTO))
print("=" * 74)
_e_s38 = res_s38.error_vs(V_EXACTO)[0]
print(f"  Error de Simpson 1/3 : {_e_s13:.6e} cm³")
print(f"  Error de Simpson 3/8 : {_e_s38:.6e} cm³")
print(f"  Cociente 3/8 ÷ 1/3   : {_e_s38 / max(_e_s13, 1e-300):.2f}×"
      f"   (teoría para f suave: 180/80 = 2.25×)")

**Cómo leer el resultado — la segunda sorpresa, y va en dirección contraria.**

- **La teoría dice que 3/8 debería ser 2.25× PEOR que 1/3** (constantes $1/80$ frente a
  $1/180$). **El cómputo da lo opuesto: 3/8 sale ≈ 3.7 veces MEJOR** (cociente ≈ 0.27).
  Con $n = 60$, Simpson 3/8 acumula ~0.071 cm³ de error frente a los ~0.263 cm³ de
  Simpson 1/3.
- **Por qué.** La comparación teórica $180$ vs $80$ solo vale si $f^{(4)}$ existe y es
  acotada. En nuestro envase no lo es. Lo que realmente decide el resultado es **cuántos
  bloques de la regla caen a caballo de un codo y cómo**, y eso depende de la aritmética
  concreta entre $n$, la posición de los quiebres y el ancho del bloque de cada regla —
  no de la constante de la cota.
- **La conclusión metodológica, que es lo que hay que llevarse:** cuando el error
  observado contradice la teoría, **lo que falla no es la teoría — es que sus hipótesis
  no se cumplen**. Distinguir "mi código tiene un bug" de "estoy aplicando una fórmula
  fuera de su dominio de validez" es una de las habilidades centrales del análisis
  numérico. Aquí es lo segundo, y lo demostramos: con malla alineada (Sección 8) ambas
  reglas caen al suelo de redondeo, tal como predice la teoría.
- **⚠️ No generalices este resultado.** Que 3/8 gane *aquí* con *este* $n$ es un accidente
  de alineación, no una propiedad del método. Con otro $n$ el orden puede invertirse otra
  vez. Es precisamente la marca de un régimen donde **el orden teórico ya no gobierna**:
  los resultados se vuelven erráticos e impredecibles. Lo verás con claridad en la
  gráfica de convergencia de la Sección 8, donde estas curvas suben y bajan en vez de
  descender limpiamente.
- **`Notas`** indica si se usó 3/8 puro o composición con 1/3. Con $n = 60$ (múltiplo de
  3) será pura.
- **Este método satisface el requisito explícito de la §6 del PDF** de implementar una
  Newton-Cotes cerrada tipo 3/8. El siguiente, Boole, cubre además la opción de *"orden
  superior"* mencionada en la misma línea.

### 5.5 Regla de Boole — Newton-Cotes cerrada de orden superior

Cubre la otra opción que ofrece la §6 del PDF: *"...o una de orden superior"*.

#### La idea

Grado 4: ajustamos un polinomio de **cuarto grado** por cada **cuatro** subintervalos
(5 puntos determinan una cuártica):

$$\int_{z_i}^{z_{i+4}} f\,dz \;\approx\; \frac{2\,\Delta z}{45}\bigl[7f_i + 32f_{i+1} + 12f_{i+2} + 32f_{i+3} + 7f_{i+4}\bigr]$$

Los pesos $[7, 32, 12, 32, 7]$ ya no son "bonitos" como los de Simpson — son el
resultado de integrar los polinomios de Lagrange de grado 4 sobre 5 nodos equiespaciados.
Al componer, los extremos compartidos entre bloques consecutivos suman $7+7 = 14$:

$$[\,7,\;32,\;12,\;32,\;\mathbf{14},\;32,\;12,\;32,\;\mathbf{14},\;\dots,\;32,\;12,\;32,\;7\,]$$

#### Condición: n múltiplo de 4

#### Error

$$E \;=\; -\frac{2(b-a)\,\Delta z^{6}}{945}\,f^{(6)}(\xi)$$

**$O(\Delta z^6)$**: al duplicar los puntos, el error cae a **1/64**. Y el **grado de
exactitud es 5**, no 4 — otra vez el grado extra de regalo, porque 4 es **par**. Esto
confirma el patrón anunciado en la sección anterior.

#### Por qué Newton-Cotes se detiene aquí en la práctica

Podríamos seguir: grado 6, 8, 10... Pero a partir de grado ≈ 8 los pesos de
Newton-Cotes **empiezan a tomar signos alternados y magnitudes crecientes**. Eso hace la
fórmula **numéricamente inestable**: se restan números grandes para obtener uno pequeño,
y la cancelación catastrófica amplifica el error de redondeo hasta destruir el
resultado. Es una manifestación del **fenómeno de Runge**, la razón por la que
interpolar con polinomios de alto grado sobre nodos *equiespaciados* es una mala idea.

Ese callejón sin salida es precisamente **lo que motiva la cuadratura de Gauss**: si los
nodos equiespaciados limitan el grado alcanzable, la salida es **dejar de exigir que los
nodos sean equiespaciados**. Es el tema de la siguiente sección.

#### Predicción para nuestro envase

Grado de exactitud 5 ≥ 2 = grado de $f = r^2$. Igual que Simpson, Boole debería ser
**exacto dentro de cada tramo**, y todo su error debería venir de los bloques que cruzan
un quiebre. Como sus bloques son más anchos (4 subintervalos frente a 2), **cruzan
quiebres con más frecuencia**. Es perfectamente posible que Boole salga **peor que
Simpson 1/3** aquí, pese a ser de orden muy superior. Si ocurre, no es un error: es la
consecuencia directa de aplicar una herramienta de alto orden a una función no suave.

In [ ]:
def regla_boole(z: np.ndarray, r: np.ndarray) -> ResultadoIntegracion:
    """Regla de Boole compuesta (Newton-Cotes cerrada de grado 4).

    Pesos [7,32,12,32,14,32,12,32,14,...,32,12,32,7] · 2Δz/45.
    Condición: n múltiplo de 4.
    """
    n, dz = _validar_malla(z, r)
    if n % 4 != 0:
        faltan = 4 - (n % 4)
        raise CondicionNoSatisfecha(
            f"La regla de Boole requiere n múltiplo de 4 y se recibió n = {n} "
            f"(n % 4 = {n % 4}).\n"
            f"  Motivo: cada polinomio de grado 4 se ajusta sobre 4 subintervalos "
            f"consecutivos (5 nodos).\n"
            f"  Soluciones: usar n = {n - (n % 4)} o n = {n + faltan}, o remuestrear "
            f"el perfil a una malla con n múltiplo de 12 (compatible con los cuatro "
            f"métodos a la vez)."
        )
    f = np.asarray(r, dtype=float) ** 2

    def _calcular() -> float:
        w = np.zeros(n + 1)
        for i in range(0, n, 4):          # bloques de 4 subintervalos
            w[i]     +=  7.0
            w[i + 1] += 32.0
            w[i + 2] += 12.0
            w[i + 3] += 32.0
            w[i + 4] +=  7.0              # los extremos compartidos acumulan 7+7=14
        return float(2.0 * dz / 45.0 * np.dot(w, f))

    integral, ms = _cronometrar(_calcular)
    return ResultadoIntegracion(
        metodo="Boole compuesto",
        familia="Newton-Cotes cerrada (grado 4)",
        n_subintervalos=n,
        n_evaluaciones=n + 1,
        a=float(z[0]), b=float(z[-1]),
        delta_z=dz,
        orden_error="O(Δz⁶)",
        grado_exactitud=5,
        integral_r2=integral,
        volumen_cm3=math.pi * integral,
        tiempo_ms=ms,
        notas="Requiere n múltiplo de 4. Bloques anchos: sensible a puntos de quiebre.",
    )


res_boole = regla_boole(Z_BASE, R_BASE)
print("=" * 74)
print("MÉTODO 4 — REGLA DE BOOLE  (Newton-Cotes de orden superior)")
print("=" * 74)
print(res_boole.informe(V_EXACTO))
print("=" * 74)
print("  Comparación entre las Newton-Cotes, con los MISMOS 61 puntos:")
for _r in (res_trap, res_s13, res_s38, res_boole):
    _ea, _er = _r.error_vs(V_EXACTO)
    print(f"     {_r.metodo:<24} {_r.orden_error:<9} error = {_ea:.6e} cm³  ({_er:.3e} %)")

**Cómo leer el resultado — la tabla comparativa al final de la celda es la clave.**

El ranking que verás, de mejor a peor, es este:

| Puesto | Método | Orden teórico | Error [cm³] |
|---|---|---|---|
| 1 | **Trapecio** | $O(\Delta z^2)$ | ≈ 0.063 |
| 2 | Simpson 3/8 | $O(\Delta z^4)$ | ≈ 0.071 |
| 3 | Simpson 1/3 | $O(\Delta z^4)$ | ≈ 0.263 |
| 4 | **Boole** | $O(\Delta z^6)$ | ≈ 0.301 |

**El ranking está exactamente invertido respecto al orden teórico.** El método de orden
2 gana; el de orden 6 pierde. Y no por poco: Boole acumula casi 5 veces más error que el
trapecio.

- **La causa es la anticipada:** los bloques de Boole abarcan 4 subintervalos, los más
  anchos de las cuatro reglas, así que **cruzan codos con más frecuencia que ninguna
  otra**. En un codo, la hipótesis "$f^{(6)}$ existe y es acotada" no se cumple — ni
  siquiera existe $f''$ — y la cota $O(\Delta z^6)$ sencillamente **no es aplicable**.
  Lo que queda es un polinomio de grado 4 intentando ajustarse a cinco puntos repartidos
  entre dos curvas distintas.
- **Hay una regularidad interesante en el ranking:** ordena los métodos por **ancho de
  bloque** (1, 3, 2, 4 subintervalos) más que por orden teórico. Cuanto más ancho el
  bloque, más codos cruza y más error acumula. Sobre integrandos quebrados, **el ancho
  del estarcido predice el error mejor que el orden de la fórmula**.
- **Esta es la observación más valiosa de toda la sección**, y merece ir textualmente al
  informe: *el orden teórico de un método solo se materializa si el integrando cumple
  las hipótesis de suavidad de su cota de error*. Sobre datos extraídos de una fotografía
  — con ruido de píxel y aristas del envase — **los métodos de orden alto no solo pierden
  su ventaja: pueden quedar por detrás de los más simples**.
- **Los cuatro tiempos de cálculo son del mismo orden** (microsegundos). El coste
  computacional **no** es un criterio de selección aquí. Lo que decide es la robustez
  frente a datos imperfectos, y en ese criterio **gana el método más humilde**.
- **Perspectiva, para no exagerar:** los cuatro errores están entre 0.06 y 0.30 cm³, es
  decir, entre el **0.2 % y el 1.0 % del margen del 5 %** (que son 30.8 cm³). Los cuatro
  son perfectamente utilizables. La discusión sobre cuál gana es **académicamente
  fascinante y prácticamente irrelevante** — y saber distinguir esas dos cosas también
  es parte del oficio.

### 5.6 Cuadratura de Gauss-Legendre

#### El cambio de perspectiva

Todas las reglas anteriores comparten una restricción autoimpuesta: **los nodos están
fijos y equiespaciados**, y lo único que se elige son los pesos. Con $m$ nodos hay $m$
pesos libres, lo que permite imponer exactitud hasta grado $m-1$.

Gauss se pregunta: *¿y si también elegimos dónde poner los nodos?*

Con $m$ nodos hay entonces $2m$ parámetros libres ($m$ posiciones + $m$ pesos), y se
puede lograr exactitud hasta grado:

$$\boxed{\;2m - 1\;}$$

**Con 3 nodos, Gauss integra exactamente polinomios de grado 5** — algo para lo que
Newton-Cotes necesita 5 nodos (Boole). Es prácticamente el doble de rendimiento por
evaluación.

#### Dónde van los nodos

Los nodos óptimos son las **raíces del polinomio de Legendre** $P_m(x)$ en $[-1, 1]$.
No son equiespaciados: se **agrupan hacia los extremos** del intervalo, que es
precisamente el patrón que evita el fenómeno de Runge.

Para $m = 3$: $x = -\sqrt{3/5},\; 0,\; +\sqrt{3/5}$, con pesos $5/9,\; 8/9,\; 5/9$.

#### El cambio de variable

Los nodos de Legendre viven en $[-1, 1]$ y nuestro envase en $[a, b]$. La transformación
afín es:

$$z = \frac{b-a}{2}x + \frac{a+b}{2}, \qquad dz = \frac{b-a}{2}\,dx$$

y por tanto:

$$\int_a^b f(z)\,dz \;=\; \frac{b-a}{2}\int_{-1}^{1} f\!\left(\tfrac{b-a}{2}x + \tfrac{a+b}{2}\right)dx \;\approx\; \frac{b-a}{2}\sum_{k=1}^{m} w_k\, f(z_k)$$

El factor $\frac{b-a}{2}$ es el **jacobiano** del cambio de variable. Olvidarlo es el
error de implementación más común en Gauss, y es fácil de detectar: el resultado sale
escalado por un factor constante, en nuestro caso $21/2 = 10.5$.

#### ⚠️ El problema práctico, y por qué importa tanto para este proyecto

Los nodos de Gauss caen en posiciones **irracionales** como $z = 10.5 - 10.5\sqrt{3/5}$.

De una fotografía **jamás** obtendremos el radio exactamente ahí: los datos vienen en
filas de píxeles equiespaciadas. **Gauss exige poder evaluar el integrando donde él
decida, y con datos tabulados eso es imposible.**

La solución obligada: **interpolar**. Construimos un spline cúbico sobre los datos
$(z_i, r_i)$ y lo evaluamos en los nodos de Gauss. Y aquí está el punto crítico:

> El resultado de Gauss deja de ser "la integral de los datos" y pasa a ser
> **"la integral del spline que ajustamos a los datos"**. Hemos introducido una segunda
> aproximación — el modelo de interpolación — encima de la cuadratura.

Por eso implementamos **dos variantes**, y la comparación entre ambas es material de
primera para el informe:

| Variante | Evalúa el radio con | Situación que representa |
|---|---|---|
| `gauss_desde_funcion` | el perfil analítico exacto | ideal irreal: el "techo" de Gauss |
| `gauss_desde_datos` | spline cúbico sobre la tabla | **lo que realmente ocurrirá con una foto** |

La §7 Etapa 5 del PDF exige *"exactamente el mismo perfil de entrada, o una
representación equivalente"*: el spline sobre la misma tabla **es** esa representación
equivalente, y la variante analítica queda como referencia diagnóstica.

In [ ]:
def _nodos_pesos_legendre(m: int) -> tuple[np.ndarray, np.ndarray]:
    """Nodos y pesos de Gauss-Legendre de m puntos en [-1, 1].

    numpy los calcula como raíces del polinomio de Legendre P_m mediante el
    autovalor de la matriz de Jacobi asociada — estable incluso para m grande.
    """
    if m < 1:
        raise CondicionNoSatisfecha(f"El número de nodos de Gauss debe ser >= 1 (m={m}).")
    x, w = np.polynomial.legendre.leggauss(m)
    return x, w


def gauss_desde_funcion(perfil: PerfilEnvase, m: int) -> ResultadoIntegracion:
    """Gauss-Legendre de m nodos evaluando el perfil ANALÍTICO.

    Es el caso ideal: podemos pedir el radio en cualquier z con precisión exacta.
    En la fase de cámara esto NO estará disponible; sirve como cota superior del
    rendimiento alcanzable por Gauss.
    """
    x, w = _nodos_pesos_legendre(m)
    a, b = perfil.a, perfil.b
    semi = (b - a) / 2.0            # jacobiano del cambio de variable
    centro = (a + b) / 2.0
    z_nodos = semi * x + centro     # mapeo [-1,1] → [a,b]

    def _calcular() -> float:
        f = perfil.radio_cuadrado(z_nodos)
        return float(semi * np.dot(w, f))

    integral, ms = _cronometrar(_calcular)
    return ResultadoIntegracion(
        metodo=f"Gauss-Legendre {m} nodos (perfil analítico)",
        familia="Cuadratura gaussiana",
        n_subintervalos=0,              # no hay subintervalos: no es una regla compuesta
        n_evaluaciones=m,
        a=float(a), b=float(b),
        delta_z=None,                   # nodos NO equiespaciados
        orden_error=f"exacta para grado ≤ {2*m - 1}",
        grado_exactitud=2 * m - 1,
        integral_r2=integral,
        volumen_cm3=math.pi * integral,
        tiempo_ms=ms,
        notas=f"Nodos en las raíces de P_{m}(x), reescalados a [a,b]. Caso ideal.",
    )


def gauss_desde_datos(z: np.ndarray, r: np.ndarray, m: int,
                      tipo_spline: str = "cubico") -> ResultadoIntegracion:
    """Gauss-Legendre de m nodos sobre un SPLINE ajustado a la tabla (z, r).

    Es el escenario realista: los datos vienen tabulados en posiciones fijas y hay
    que interpolar para evaluar el integrando en los nodos de Gauss.

    IMPORTANTE: el resultado es la integral del SPLINE, no la de los datos. La
    calidad del spline pasa a formar parte del error total.
    """
    z = np.asarray(z, dtype=float)
    r = np.asarray(r, dtype=float)
    if len(z) < 4 and tipo_spline == "cubico":
        raise CondicionNoSatisfecha(
            f"Un spline cúbico requiere al menos 4 puntos y se recibieron {len(z)}.\n"
            f"  Solución: aumentar n, o usar tipo_spline='lineal'."
        )
    grado = {"lineal": 1, "cuadratico": 2, "cubico": 3}[tipo_spline]
    spline = interpolate.InterpolatedUnivariateSpline(z, r, k=grado, ext="const")

    x, w = _nodos_pesos_legendre(m)
    a, b = float(z[0]), float(z[-1])
    semi = (b - a) / 2.0
    centro = (a + b) / 2.0
    z_nodos = semi * x + centro

    def _calcular() -> float:
        f = spline(z_nodos) ** 2
        return float(semi * np.dot(w, f))

    integral, ms = _cronometrar(_calcular)
    return ResultadoIntegracion(
        metodo=f"Gauss-Legendre {m} nodos (spline sobre datos)",
        familia="Cuadratura gaussiana",
        n_subintervalos=len(z) - 1,
        n_evaluaciones=m,
        a=a, b=b,
        delta_z=None,
        orden_error=f"exacta para grado ≤ {2*m - 1} (del spline)",
        grado_exactitud=2 * m - 1,
        integral_r2=integral,
        volumen_cm3=math.pi * integral,
        tiempo_ms=ms,
        notas=(f"Spline {tipo_spline} (k={grado}) sobre {len(z)} puntos, evaluado en "
               f"los {m} nodos de Gauss. Este es el escenario de la fase de cámara."),
    )


# --- Ejecución con m = 5 nodos ----------------------------------------
M_GAUSS = 5
res_gauss_fn = gauss_desde_funcion(ENVASE, M_GAUSS)
res_gauss_dt = gauss_desde_datos(Z_BASE, R_BASE, M_GAUSS)

_x5, _w5 = _nodos_pesos_legendre(M_GAUSS)
_z5 = (ENVASE.b - ENVASE.a) / 2 * _x5 + (ENVASE.a + ENVASE.b) / 2

print("=" * 74)
print(f"MÉTODO 5 — CUADRATURA DE GAUSS-LEGENDRE ({M_GAUSS} nodos)")
print("=" * 74)
print(f"  Nodos en [-1, 1] : {_x5}")
print(f"  Pesos            : {_w5}")
print(f"  Suma de pesos    : {_w5.sum():.12f}   (debe ser exactamente 2 = longitud de [-1,1])")
print(f"  Nodos en [a, b]  : {_z5}  cm")
print(f"  Jacobiano (b−a)/2: {(ENVASE.b - ENVASE.a) / 2:.6f}")
print("=" * 74)
print("\n--- Variante A: evaluando el perfil ANALÍTICO (ideal) ---")
print(res_gauss_fn.informe(V_EXACTO))
print("\n--- Variante B: evaluando un SPLINE sobre la tabla (realista) ---")
print(res_gauss_dt.informe(V_EXACTO))
print("=" * 74)

**Cómo leer el resultado:**

- **`Suma de pesos = 2.000000000000`** — verificación imprescindible. Los pesos de
  Gauss-Legendre deben sumar la longitud del intervalo canónico $[-1,1]$, que es 2.
  Equivale a decir que el método integra exactamente la función constante $f=1$. Si esta
  suma no diera 2, habría un error en la generación de nodos y **todo lo demás sería
  inválido**. Es la comprobación de sanidad más barata y más útil de Gauss.
- **Los nodos NO están equiespaciados**: observa cómo se agrupan hacia $\pm 1$ y dejan
  un hueco mayor en el centro. Esa distribución es lo que evita el fenómeno de Runge y
  hace posible el grado de exactitud $2m-1$.
- **Solo 5 evaluaciones** frente a las 61 de los métodos anteriores. Gauss es
  **doce veces más barato en datos**. Si cada evaluación costara un experimento de
  laboratorio, esa diferencia sería decisiva.
- **⚠️ Gauss con 5 nodos es, con diferencia, el PEOR método de todo el notebook:**
  ≈ 3.9 cm³ de error, un **0.63 %** — unas **13 veces peor** que Boole y **62 veces peor**
  que el trapecio. Consume él solo el **12.7 % del margen del 5 %**, mientras que los
  cuatro Newton-Cotes se quedan por debajo del 1 %.
- **Por qué falla tan estrepitosamente.** Con 5 nodos, Gauss es exacto hasta grado 9 —
  muy por encima del grado 2 de $f = r^2$. Pero **nuestro integrando no es un polinomio
  global**: es cuadrático *por tramos*, con cuatro esquinas. Ningún polinomio de grado 9,
  por bien elegidos que estén sus nodos, puede reproducir una función quebrada. Y Gauss
  es una regla **global**, no compuesta: no subdivide el intervalo, intenta capturar los
  21 cm de envase con **un solo polinomio de grado 9** y 5 muestras. Con 5 puntos es
  literalmente imposible saber que hay un hombro en $z=15$.
- **La comparación entre las dos variantes da un resultado inesperado:** la variante B
  (spline, ≈ 3.01 cm³) sale **algo MEJOR** que la variante A (perfil exacto, ≈ 3.91 cm³),
  pese a que B añade un error de interpolación que A no tiene.
- **Esto no significa que interpolar mejore las cosas.** Es una **cancelación
  fortuita**: el spline cúbico *suaviza* los codos del envase, redondeando el hombro y
  el pie. Ese sesgo de modelado resulta tener signo opuesto al error de Gauss sobre la
  función quebrada, y los dos se compensan parcialmente. Nada lo garantiza — con otro
  envase, otro $m$ u otro tipo de spline, los dos errores podrían sumarse en vez de
  restarse.
- **La lección correcta:** cuando dos errores independientes son del mismo orden de
  magnitud, su suma es impredecible en signo y magnitud. **Que un resultado salga mejor
  no lo hace más fiable.** Aquí ambas variantes están en ~0.5-0.6 %, y esa es la cifra
  honesta del rendimiento de Gauss-5 sobre este problema; cuál de las dos sale encima es
  ruido.
- **Conclusión para el proyecto — y va contra la intuición:** con datos de imagen, Gauss
  **no solo no es superior: es el peor candidato**. Su ventaja teórica se paga con dos
  facturas que los Newton-Cotes no pagan: (1) depende de una interpolación, porque no
  puede evaluar donde no hay datos; y (2) al ser una regla global, es **especialmente
  vulnerable a la falta de suavidad**. Con 61 puntos disponibles y una foto ruidosa,
  **el trapecio o Simpson sobre los datos crudos son la opción sensata**.
- **Gauss brilla** cuando evaluar el integrando es caro (cada evaluación es una
  simulación o un experimento) y la función es suave. **Nuestro escenario es exactamente
  el contrario**: evaluaciones gratis, función quebrada. Reportarlo es obligatorio por el
  enunciado; usarlo como valor principal sería un error de ingeniería.

### 5.7 ¿Y si le damos más nodos a Gauss?

La reacción natural al resultado anterior es: *"5 nodos son pocos, subamos a 20 o 50"*.

**Qué hace esta celda:** barre $m$ desde 2 hasta 160 nodos y mide el error.

La §6 del PDF pide analizar la *"influencia del orden de cuadratura"*. Esto lo hace, y
el resultado es instructivo precisamente porque **no se comporta como uno espera**.

In [ ]:
filas_g = []
for m in [2, 3, 5, 8, 12, 20, 40, 80, 160]:
    rg = gauss_desde_funcion(ENVASE, m)
    e_abs, e_rel = rg.error_vs(V_EXACTO)
    filas_g.append({
        "m (nodos)": m,
        "Grado exacto (2m−1)": 2 * m - 1,
        "V [cm³]": rg.volumen_cm3,
        "Error abs. [cm³]": e_abs,
        "Error rel. [%]": e_rel,
        "% del margen del 5%": 100 * e_abs / (0.05 * V_EXACTO),
        "¿Pasa el 5%?": "✓" if e_rel < 5 else "✗ NO",
    })
GAUSS_BARRIDO = pd.DataFrame(filas_g)

print("=" * 78)
print("INFLUENCIA DEL ORDEN DE CUADRATURA EN GAUSS-LEGENDRE")
print("=" * 78)
display(GAUSS_BARRIDO)

# ¿Es monótono el error al aumentar m?
_errs = GAUSS_BARRIDO['Error abs. [cm³]'].to_numpy()
_subidas = [(int(GAUSS_BARRIDO['m (nodos)'].iloc[i]), int(GAUSS_BARRIDO['m (nodos)'].iloc[i+1]))
            for i in range(len(_errs) - 1) if _errs[i+1] > _errs[i]]
print("=" * 78)
if _subidas:
    print('  ⚠ El error NO decrece monótonamente al añadir nodos. Empeora al pasar de:')
    for m1, m2 in _subidas:
        print(f'       m = {m1}  →  m = {m2}')
else:
    print('  El error decrece monótonamente.')

_trap_err = res_trap.error_vs(V_EXACTO)[0]
_m_necesarios = GAUSS_BARRIDO[GAUSS_BARRIDO['Error abs. [cm³]'] < _trap_err]
print('=' * 78)
if len(_m_necesarios):
    _m = int(_m_necesarios['m (nodos)'].iloc[0])
    print(f'  Gauss necesita m ≥ {_m} nodos para igualar al TRAPECIO con 61 puntos')
    print(f'  (error del trapecio: {_trap_err:.4e} cm³)')
else:
    print(f'  Ni con 160 nodos Gauss alcanza la precisión del trapecio con 61 puntos')
    print(f'  (error del trapecio: {_trap_err:.4e} cm³)')
print('=' * 78)

**Cómo leer el resultado — es contundente:**

- **Con $m = 2$ y $m = 3$ nodos, Gauss FALLA el criterio del 5 % del reto** (≈ 5.1 % y
  ≈ 5.3 % de error). Si alguien implementara Gauss de bajo orden como método principal,
  **suspendería la prueba de campo aun con una fotografía perfecta**.
- **El error NO decrece monótonamente.** Añadir nodos a veces **empeora** el resultado:
  de $m=2$ a $m=3$ sube, de $m=5$ a $m=8$ sube, de $m=12$ a $m=20$ sube. Esto es
  desconcertante si uno espera que "más puntos = mejor", y es la firma inequívoca de un
  método **fuera de su régimen de validez**. La convergencia de Gauss está garantizada
  para funciones analíticas; para una función con esquinas, degenera a una convergencia
  lenta y errática.
- **Gauss necesita ≈ 80 nodos para igualar lo que el trapecio consigue con 61 puntos.**
  Es decir: el método teóricamente más eficiente en evaluaciones necesita **más
  evaluaciones que el más elemental** para llegar al mismo sitio. La ventaja de
  "grado $2m-1$ con $m$ nodos", que sobre una función suave sería aplastante, aquí se
  evapora por completo.
- **La moraleja, que aplica mucho más allá de este laboratorio:**

> Un método numérico no es "bueno" o "malo" en abstracto: es bueno **para una clase de
> problemas**. Gauss-Legendre es extraordinario para integrandos suaves y evaluaciones
> caras. Aplicado a una función quebrada muestreada densamente, es la peor elección
> disponible. **Elegir el método sin mirar la estructura del problema es el error de
> ingeniería que este notebook existe para prevenir.**

- **Cómo se arreglaría** (fuera del alcance de esta fase): aplicar Gauss **por tramos**,
  es decir, una cuadratura gaussiana independiente en cada tramo suave del envase. Eso
  convierte una regla global en una compuesta y recupera su precisión. Es la misma idea
  de "alinear la malla con la estructura del problema" que la Sección 8 desarrolla para
  los Newton-Cotes, y es exactamente lo que hace `scipy.integrate.quad` con su argumento
  `points`.

---
## 6. Validación de condiciones y mensajes claros — §8 del PDF

El enunciado exige literalmente:

> *"Mensajes claros cuando los datos no cumplen las condiciones requeridas por un
> método."*

**Qué hace esta celda:** provoca **a propósito** los tres fallos posibles y muestra el
mensaje que produce cada uno.

**Por qué probar los fallos es tan importante como probar los aciertos.** El día de la
prueba de campo, el número de puntos vendrá determinado por la altura en píxeles del
envase en la foto — un número arbitrario que nadie controla. Si sale $n = 437$, Simpson,
3/8 y Boole fallan los tres. Lo que decide si la aplicación sirve o no en ese momento no
es que calcule bien cuando todo va bien; es **qué hace cuando algo no encaja**.

Un buen mensaje de error tiene tres partes, y las tres son obligatorias:

1. **Qué falló** — el dato concreto y el requisito concreto.
2. **Por qué** — la razón matemática, no solo la regla.
3. **Cómo arreglarlo** — el valor válido más cercano, o la alternativa.

Un mensaje que solo dice *"Error: n inválido"* obliga al usuario a leer el código
fuente. Eso no es un mensaje de error, es una rendición.

Los tres escenarios que provocamos:

| # | Escenario | Método que debe rechazar |
|---|---|---|
| A | $n$ impar | Simpson 1/3 |
| B | $n$ no múltiplo de 4 | Boole |
| C | malla no uniforme | todas las Newton-Cotes |

El escenario C es el más relevante para la fase de cámara: si el contorno se extrae con
huecos — por un reflejo, una etiqueta o una sombra que rompen la segmentación — las
alturas dejan de estar equiespaciadas. **El sistema debe detectarlo y decirlo, nunca
integrar igual y devolver un número con aspecto razonable.**

In [ ]:
print("=" * 74)
print("PRUEBA DELIBERADA DE CONDICIONES NO SATISFECHAS")
print("=" * 74)

# --- Escenario A: n impar -> Simpson 1/3 debe rechazar -----------------
print("\n▸ ESCENARIO A — n impar (n = 61)")
print("-" * 74)
z_impar, r_impar = muestrear_perfil(ENVASE, 61)
try:
    regla_simpson_13(z_impar, r_impar)
    print("  ✗ FALLO DE LA PRUEBA: debió lanzar excepción y no lo hizo.")
except CondicionNoSatisfecha as e:
    print(f"  ✓ Rechazado correctamente:\n\n{e}")

# El trapecio, en cambio, acepta cualquier n.
res_impar = regla_trapecio(z_impar, r_impar)
print(f"\n  → El trapecio SÍ acepta n = 61: V = {res_impar.volumen_cm3:.6f} cm³")
print(f"    (el trapecio es el respaldo universal cuando n viene impuesto)")

# --- Escenario B: n no múltiplo de 4 -> Boole debe rechazar ------------
print("\n\n▸ ESCENARIO B — n = 30, no es múltiplo de 4")
print("-" * 74)
z_30, r_30 = muestrear_perfil(ENVASE, 30)
try:
    regla_boole(z_30, r_30)
    print("  ✗ FALLO DE LA PRUEBA: debió lanzar excepción.")
except CondicionNoSatisfecha as e:
    print(f"  ✓ Rechazado correctamente:\n\n{e}")

print(f"\n  → Simpson 1/3 sí acepta n = 30 (es par): "
      f"V = {regla_simpson_13(z_30, r_30).volumen_cm3:.6f} cm³")

# --- Escenario C: malla no uniforme -----------------------------------
print("\n\n▸ ESCENARIO C — malla NO uniforme (contorno con huecos)")
print("-" * 74)
# Simulamos la pérdida de tres filas del contorno, como ocurriría si un reflejo
# o una etiqueta rompieran la segmentación en esa franja del envase.
z_hueco = np.delete(Z_BASE, [20, 21, 22])
r_hueco = np.delete(R_BASE, [20, 21, 22])
print(f"  Puntos: {len(Z_BASE)} → {len(z_hueco)} (se perdieron 3 filas del contorno)")
print(f"  Espaciamientos únicos presentes: {np.unique(np.round(np.diff(z_hueco), 6))} cm")
try:
    regla_trapecio(z_hueco, r_hueco)
    print("  ✗ FALLO DE LA PRUEBA: debió lanzar excepción.")
except CondicionNoSatisfecha as e:
    print(f"  ✓ Rechazado correctamente:\n\n{e}")

print("\n" + "=" * 74)
print("Las tres condiciones se detectan y se comunican con un mensaje accionable.")
print("=" * 74)

**Cómo leer el resultado:**

Las tres marcas ✓ confirman que el sistema **no calcula cuando no debe**. Fíjate en que
cada mensaje nombra el valor recibido, explica la razón geométrica y propone
alternativas concretas.

Observa también los dos **caminos de recuperación** que quedan demostrados:

- Con $n = 61$ (impar), el **trapecio sigue funcionando**. Es el respaldo universal: sin
  restricciones sobre $n$. En la aplicación final, si ningún método de orden superior
  aplica, el trapecio siempre da una respuesta válida.
- Con $n = 30$, Boole falla pero **Simpson 1/3 funciona**. La estrategia correcta no es
  "elegir un método" sino **probar los cuatro y reportar los que apliquen**, avisando de
  los que no. Eso es justo lo que hace la tabla de la sección siguiente.

El escenario C merece atención especial para la fase de cámara: la malla no uniforme
por pérdida de filas del contorno es un fallo **probable**, no hipotético. La solución
allí será **remuestrear por interpolación** a una malla uniforme con $n$ múltiplo de 12
antes de integrar — y declararlo explícitamente como parte del procedimiento, porque
introduce su propio error.

---
## 7. Tabla comparativa de los cuatro métodos — §7 Etapa 5 y §8

**Qué hace esta celda:** ejecuta todos los métodos sobre **la misma tabla de datos** y
consolida los resultados.

La §7 Etapa 5 obliga a reportar, para cada método: número de subintervalos o puntos,
intervalo $[a,b]$, $\Delta z$, orden o número de nodos, integral estimada, volumen
estimado y tiempo de cálculo. La §7 Etapa 6 añade diferencia absoluta y error relativo
porcentual. **Todo eso está en las columnas de abajo.**

`ejecutar_todos` captura `CondicionNoSatisfecha` por método: si uno no aplica, se
registra con su motivo y **los demás siguen ejecutándose**. Un método incompatible no
debe tumbar el análisis completo.

In [ ]:
def ejecutar_todos(perfil: PerfilEnvase, n: int, m_gauss: int = 5,
                   verbose: bool = True) -> tuple[list[ResultadoIntegracion], list[tuple[str, str]]]:
    """Aplica todos los métodos a la misma tabla y recolecta resultados y fallos.

    Devuelve (resultados_exitosos, [(metodo, motivo_del_fallo), ...]).
    """
    z, r = muestrear_perfil(perfil, n)
    metodos: list[tuple[str, Callable[[], ResultadoIntegracion]]] = [
        ("Trapecio",            lambda: regla_trapecio(z, r)),
        ("Simpson 1/3",         lambda: regla_simpson_13(z, r)),
        ("Simpson 3/8",         lambda: regla_simpson_38(z, r)),
        ("Boole",               lambda: regla_boole(z, r)),
        ("Gauss (analítico)",   lambda: gauss_desde_funcion(perfil, m_gauss)),
        ("Gauss (spline)",      lambda: gauss_desde_datos(z, r, m_gauss)),
    ]
    exitosos: list[ResultadoIntegracion] = []
    fallidos: list[tuple[str, str]] = []
    for nombre, fn in metodos:
        try:
            exitosos.append(fn())
        except CondicionNoSatisfecha as e:
            fallidos.append((nombre, str(e).split("\n")[0]))
            if verbose:
                print(f"  ⚠ {nombre}: no aplicable — {str(e).splitlines()[0]}")
    return exitosos, fallidos


def tabla_comparativa(resultados: Sequence[ResultadoIntegracion],
                      v_referencia: float) -> pd.DataFrame:
    """Consolida los resultados con todos los campos que exige el enunciado."""
    filas = []
    for res in resultados:
        e_abs, e_rel = res.error_vs(v_referencia)
        filas.append({
            "Método": res.metodo,
            "Familia": res.familia,
            "n": res.n_subintervalos if res.n_subintervalos else "—",
            "Evals": res.n_evaluaciones,
            "a [cm]": res.a,
            "b [cm]": res.b,
            "Δz [cm]": res.delta_z if res.delta_z is not None else np.nan,
            "Orden / exactitud": res.orden_error,
            "I = ∫r²dz [cm³]": res.integral_r2,
            "V = πI [cm³]": res.volumen_cm3,
            "V [mL]": res.volumen_ml,
            "Err. abs. [cm³]": e_abs,
            "Err. rel. [%]": e_rel,
            "Tiempo [ms]": res.tiempo_ms,
        })
    return pd.DataFrame(filas)


print("=" * 74)
print(f"COMPARACIÓN DE MÉTODOS — n = {N_BASE}, Δz = {DZ_BASE:.6f} cm, "
      f"Gauss con {M_GAUSS} nodos")
print("=" * 74)
RESULTADOS, FALLIDOS = ejecutar_todos(ENVASE, N_BASE, M_GAUSS)
TABLA_COMP = tabla_comparativa(RESULTADOS, V_EXACTO)

print(f"\n  Valor de referencia (exacto): V = {V_EXACTO:.10f} cm³")
print(f"  Métodos aplicables: {len(RESULTADOS)}   |   No aplicables: {len(FALLIDOS)}\n")

display(TABLA_COMP[["Método", "n", "Evals", "Δz [cm]", "Orden / exactitud",
                    "I = ∫r²dz [cm³]", "V = πI [cm³]"]])
print()
display(TABLA_COMP[["Método", "V [mL]", "Err. abs. [cm³]", "Err. rel. [%]", "Tiempo [ms]"]]
        .sort_values("Err. abs. [cm³]")
        .reset_index(drop=True))

_mejor = TABLA_COMP.loc[TABLA_COMP["Err. abs. [cm³]"].idxmin()]
_peor = TABLA_COMP.loc[TABLA_COMP["Err. abs. [cm³]"].idxmax()]
print("\n" + "=" * 74)
print(f"  Más preciso : {_mejor['Método']}  →  {_mejor['Err. rel. [%]']:.4e} %")
print(f"  Menos preciso: {_peor['Método']}  →  {_peor['Err. rel. [%]']:.4e} %")
print(f"  Rango de volúmenes: [{TABLA_COMP['V [mL]'].min():.4f}, "
      f"{TABLA_COMP['V [mL]'].max():.4f}] mL  "
      f"(dispersión: {TABLA_COMP['V [mL]'].max() - TABLA_COMP['V [mL]'].min():.4f} mL)")
print(f"  Criterio del reto: 5 % de {V_EXACTO:.2f} cm³ = "
      f"±{0.05 * V_EXACTO:.2f} cm³")
print("=" * 74)

**Cómo leer el resultado — esta es la tabla que va al informe.**

- **La primera tabla** documenta los *parámetros* de cada método: $n$, evaluaciones,
  $\Delta z$, orden y el par (integral, volumen). Es el registro auditable que pide la
  §7 Etapa 5. Nota que Gauss muestra `Δz = NaN`: **no tiene un Δz**, porque sus nodos no
  son equiespaciados. Es una propiedad del método, no un dato faltante, y conviene
  señalarlo así en el informe en lugar de inventar un valor.
- **La segunda tabla** está ordenada por error creciente: el mejor método arriba.
- **El ranking real, de mejor a peor, invierte la jerarquía teórica:**

  | Puesto | Método | Orden teórico | Error [cm³] | % del margen del 5 % |
  |---|---|---|---|---|
  | 1 | Trapecio | $O(\Delta z^2)$ | ≈ 0.063 | 0.21 % |
  | 2 | Simpson 3/8 | $O(\Delta z^4)$ | ≈ 0.071 | 0.23 % |
  | 3 | Simpson 1/3 | $O(\Delta z^4)$ | ≈ 0.263 | 0.85 % |
  | 4 | Boole | $O(\Delta z^6)$ | ≈ 0.301 | 0.97 % |
  | 5 | Gauss 5 nodos | grado 9 | ≈ 3.91 | **12.69 %** |

- **Los cuatro Newton-Cotes consumen menos del 1 % del margen del reto.** Para ellos, la
  elección del método es efectivamente irrelevante: la diferencia entre el mejor y el
  peor es de 0.24 cm³, sobre un margen permitido de 30.8 cm³.
- **Gauss con 5 nodos es la excepción y merece atención:** consume el **12.7 %** del
  margen él solo. No es despreciable, y con menos nodos ni siquiera pasaría el criterio
  (lo verás en la celda 5.7). **Es el único método del notebook cuya elección sí podría
  costarte la prueba de campo.**

> **Conclusión de ingeniería.** Cualquiera de los cuatro Newton-Cotes cumple el criterio
> del 5 % con enorme holgura, y entre ellos la elección es indiferente. Lo que decidirá
> el éxito o el fracaso en la prueba de campo es la **calidad del perfil $(z_i, r_i)$** —
> la calibración de la escala píxel→cm y la extracción del contorno — no la fórmula de
> cuadratura.
>
> Con una excepción que sí es una decisión de diseño: **no usar Gauss de bajo orden como
> método principal**. Se reporta porque el enunciado lo exige, pero el valor que se
> entrega debe venir de una regla compuesta sobre los datos crudos.
>
> Esto no invalida el laboratorio: lo *cuantifica*. Saber dónde **no** está el cuello de
> botella es tan valioso como saber dónde sí está, porque dice dónde invertir el esfuerzo
> restante del proyecto.

---
## 8. Estudio de convergencia: el efecto de Δz y del número de puntos — §6

La §6 del PDF pide analizar, para el trapecio, *"efecto de Δz y del número de puntos"*, y
para Simpson, *"condiciones sobre n y comportamiento del perfil"*. Esta sección lo hace
de forma cuantitativa.

### Qué vamos a medir

Ejecutamos todos los métodos para muchos valores de $n$ y graficamos el error frente a
$\Delta z$ en **escala log-log**. Esa elección de ejes no es estética: si
$E \approx C\,\Delta z^{p}$, tomando logaritmos

$$\log E \;=\; \log C \;+\; p\,\log \Delta z$$

la relación se vuelve una **recta cuya pendiente es exactamente el orden $p$**. Medir la
pendiente por regresión lineal es, por tanto, **medir el orden de convergencia
empíricamente**. Podemos entonces contrastarlo con el orden teórico y ver si la teoría se
cumple.

### Las dos mallas que vamos a comparar — y por qué

Aquí está el experimento central del notebook. Ejecutamos el barrido dos veces:

- **Malla ciega** — $n$ múltiplos de 12, sin más. Los nodos caen donde caigan; los
  puntos de quiebre del envase quedan generalmente **entre** nodos.
- **Malla alineada** — los quiebres caen **exactamente sobre nodos**. Se logra integrando
  cada tramo por separado, con su propia malla, y sumando. Así ningún bloque de Simpson
  o Boole cruza jamás una esquina, y **dentro de cada tramo el integrando es un polinomio
  de grado 2, perfectamente suave**.

**La predicción:** con malla ciega, los métodos de orden alto quedarán **muy por debajo
de su orden teórico**, porque las hipótesis de suavidad se violan en las esquinas. Con
malla alineada, deberían alcanzar su orden teórico — o incluso el error de redondeo
directamente, ya que Simpson y Boole son *exactos* para cuadráticas.

Si esta predicción se cumple, queda demostrado algo aplicable a cualquier problema real:

> **No basta con elegir un método de orden alto. Hay que darle una malla que respete la
> estructura del problema.** Alinear la malla con las singularidades del integrando es
> una técnica mucho más rentable que subir el orden de la regla.

⏱️ Esta celda tarda unos segundos: ejecuta ~200 integraciones.

In [ ]:
def integrar_por_tramos(perfil: PerfilEnvase, metodo: Callable, n_por_tramo: int,
                        **kwargs) -> float:
    """Integra cada tramo con su propia malla y suma. MALLA ALINEADA.

    Al usar los límites de cada tramo como extremos de su malla, los puntos de
    quiebre caen SIEMPRE sobre nodos y ningún bloque de la regla los cruza. Dentro
    de cada tramo el integrando es un polinomio de grado 2 (r lineal, elevado al
    cuadrado), y por tanto suave.

    Es la estrategia estándar frente a integrandos definidos a trozos, y la misma
    idea que usa scipy.integrate.quad con su argumento `points`.
    """
    total = 0.0
    for tramo in perfil.tramos:
        z_t = np.linspace(tramo.z_ini, tramo.z_fin, n_por_tramo + 1)
        r_t = tramo.radio(z_t)
        total += metodo(z_t, r_t, **kwargs).integral_r2
    return total


# ----------------------------------------------------------------------
#  BARRIDO A — malla ciega (los quiebres caen entre nodos)
# ----------------------------------------------------------------------
NS = np.array([12, 24, 48, 96, 192, 384, 768, 1536, 3072, 6144])
registros_ciega: list[dict] = []

for n in NS:
    z_n, r_n = muestrear_perfil(ENVASE, int(n))
    dz_n = (ENVASE.b - ENVASE.a) / n
    for nombre, fn in [("Trapecio", regla_trapecio),
                       ("Simpson 1/3", regla_simpson_13),
                       ("Simpson 3/8", regla_simpson_38),
                       ("Boole", regla_boole)]:
        try:
            res = fn(z_n, r_n)
            registros_ciega.append({
                "Método": nombre, "n": int(n), "Δz": dz_n,
                "V": res.volumen_cm3,
                "Error": abs(res.volumen_cm3 - V_EXACTO),
                "Err rel [%]": 100 * abs(res.volumen_cm3 - V_EXACTO) / V_EXACTO,
            })
        except CondicionNoSatisfecha:
            pass

CONV_CIEGA = pd.DataFrame(registros_ciega)

# ----------------------------------------------------------------------
#  BARRIDO B — malla alineada con los puntos de quiebre
# ----------------------------------------------------------------------
NS_TRAMO = np.array([12, 24, 48, 96, 192, 384, 768])
registros_alin: list[dict] = []

for n_t in NS_TRAMO:
    # Δz efectivo medio: altura total dividida entre el total de subintervalos
    n_total = int(n_t) * len(ENVASE.tramos)
    dz_eff = ENVASE.altura / n_total
    for nombre, fn in [("Trapecio", regla_trapecio),
                       ("Simpson 1/3", regla_simpson_13),
                       ("Simpson 3/8", regla_simpson_38),
                       ("Boole", regla_boole)]:
        try:
            I = integrar_por_tramos(ENVASE, fn, int(n_t))
            V = math.pi * I
            registros_alin.append({
                "Método": nombre, "n_por_tramo": int(n_t), "n_total": n_total,
                "Δz": dz_eff, "V": V,
                "Error": max(abs(V - V_EXACTO), 1e-16),   # piso: evitar log(0) al graficar
                "Err rel [%]": 100 * abs(V - V_EXACTO) / V_EXACTO,
            })
        except CondicionNoSatisfecha:
            pass

CONV_ALIN = pd.DataFrame(registros_alin)


# Umbral por debajo del cual un "error" ya no mide el método, sino el redondeo.
# Con V ≈ 617 y eps ≈ 2.2e-16, un solo redondeo vale ~1.4e-13; sumando miles de
# términos el ruido acumulado llega holgadamente a ~1e-11. Ajustar la regresión
# sobre esos puntos daría pendientes sin significado (incluso negativas).
UMBRAL_REDONDEO = 1e-10


def orden_observado(df: pd.DataFrame, metodo: str) -> float:
    """Pendiente de log(Error) vs log(Δz) = orden de convergencia empírico.

    Devuelve NaN si no quedan al menos 2 puntos por encima del suelo de redondeo:
    en ese caso el método fue exacto salvo aritmética finita y no hay ley de
    potencia que medir. NaN aquí significa "demasiado bueno para medirlo", no
    "falló".
    """
    sub = df[(df["Método"] == metodo) & (df["Error"] > UMBRAL_REDONDEO)]
    if len(sub) < 2:
        return float("nan")
    return float(np.polyfit(np.log(sub["Δz"]), np.log(sub["Error"]), 1)[0])


TEORICO = {"Trapecio": 2, "Simpson 1/3": 4, "Simpson 3/8": 4, "Boole": 6}

print("=" * 78)
print("ORDEN DE CONVERGENCIA OBSERVADO  (pendiente de log E vs log Δz)")
print("=" * 78)
print(f"{'Método':<16}{'Teórico':>9}{'Malla ciega':>15}{'Malla alineada':>17}   Diagnóstico")
print("-" * 78)
for met, p_teo in TEORICO.items():
    p_ciega = orden_observado(CONV_CIEGA, met)
    p_alin = orden_observado(CONV_ALIN, met)
    if math.isnan(p_alin):
        diag = "EXACTO (error en el suelo de redondeo)"
    elif abs(p_alin - p_teo) < 0.6:
        diag = "teoría confirmada"
    else:
        diag = "limitado por otra causa"
    print(f"{met:<16}{p_teo:>9}{p_ciega:>15.2f}{p_alin:>17.2f}   {diag}")
print("=" * 78)

**Cómo leer el resultado — la tabla más importante del notebook.**

Compara las tres columnas numéricas para cada método:

- **Columna `Teórico`** — el orden que promete la cota de error de la fórmula.
- **Columna `Malla ciega`** — el orden realmente observado cuando los quiebres del
  envase caen entre nodos.
- **Columna `Malla alineada`** — el orden observado cuando los quiebres caen sobre nodos.

**Qué deberías ver — y es un resultado de libro de texto:**

| Método | Teórico | Malla ciega | Malla alineada |
|---|---|---|---|
| Trapecio | 2 | ≈ **1.97** | ≈ **2.00** |
| Simpson 1/3 | 4 | ≈ **1.98** | EXACTO (suelo) |
| Simpson 3/8 | 4 | ≈ **2.06** | EXACTO (suelo) |
| Boole | 6 | ≈ **1.81** | EXACTO (suelo) |

1. **Con malla ciega, los CUATRO métodos convergen a orden ≈ 2**, sin importar que su
   orden teórico sea 2, 4 o 6. **Todos colapsan al mismo orden.** No es una coincidencia
   de estos datos: es un resultado general del análisis numérico. Cuando el integrando es
   continuo pero no derivable ($C^0$ pero no $C^1$), **el orden de convergencia de
   cualquier regla de Newton-Cotes compuesta queda limitado a 2** por los subintervalos
   que contienen las singularidades. El eslabón más débil fija el ritmo de toda la
   cadena, y ese eslabón son los codos.
2. **La consecuencia es demoledora para la intuición habitual:** pagar por una regla de
   orden 6 y recibir convergencia de orden 2 significa que **todo el trabajo extra de
   Boole se desperdicia**. Y como además su constante de error es peor y cruza más codos,
   termina dando un resultado *peor* que el trapecio. Orden superior, coste superior,
   precisión inferior.
3. **Con malla alineada, Simpson 1/3, Simpson 3/8 y Boole caen directamente al suelo de
   redondeo** (~1e-13, reportado como `EXACTO`). No es que mejoren: es que **aciertan el
   valor exacto**. Tal como predijimos, dentro de cada tramo $f = r^2$ es un polinomio de
   grado 2, y los tres tienen grado de exactitud ≥ 3, así que lo integran **exactamente**.
   El error residual de 1e-13 es solo aritmética de punto flotante.
4. **El trapecio mantiene orden 2 en ambas columnas** — es el único que no mejora, porque
   su grado de exactitud (1) es menor que el grado del integrando (2), así que ni
   alineado puede ser exacto. Pero fíjate en el contraste: **es el único método al que
   las esquinas no le quitan nada**, porque no tenía orden alto que perder. *El método
   más humilde resultó el más robusto.*

**El resultado central de esta sección, en una frase:**

> Alinear la malla con los quiebres llevó a Simpson y Boole de **orden 2 efectivo a
> exactitud total**. Cambiar de fórmula no habría logrado nada de eso. **La estructura de
> la malla importó más que el orden del método** — por varios órdenes de magnitud.

**La conclusión, y por qué trasciende este laboratorio:**

> El orden teórico de un método es una **promesa condicionada** a que el integrando sea
> suficientemente suave. Cuando la función tiene esquinas — y los envases reales las
> tienen, y las fotos añaden ruido encima — esa promesa **no se cumple**, y un método de
> orden 6 puede rendir como uno de orden 2.
>
> Alinear la malla con la estructura del problema resultó **más eficaz que subir el
> orden de la regla**. En la fase de cámara esto se traduce en una recomendación
> concreta: detectar los quiebres del perfil (hombro, pie) e integrar por tramos, en
> lugar de buscar cuadraturas más sofisticadas.

### 8.1 Gráfica de convergencia

**Qué hace esta celda:** dibuja el error frente a $\Delta z$ en log-log, para ambas
mallas, con rectas de referencia de pendiente 2, 4 y 6.

Las rectas punteadas grises son las **pendientes teóricas**. Una curva de error paralela
a la recta de pendiente 4 confirma orden 4; una curva más horizontal que su referencia
indica que el método **no está alcanzando su orden**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
colores = {"Trapecio": "#1f77b4", "Simpson 1/3": "#2ca02c",
           "Simpson 3/8": "#ff7f0e", "Boole": "#d62728"}

# Marcadores distintos: en el panel derecho tres curvas caen al mismo suelo y
# se superponen; sin marcadores diferenciados serían indistinguibles.
marcadores = {"Trapecio": "o", "Simpson 1/3": "s", "Simpson 3/8": "^", "Boole": "D"}

for ax, df, titulo in [
    (axes[0], CONV_CIEGA, "Malla CIEGA\n(quiebres entre nodos)"),
    (axes[1], CONV_ALIN, "Malla ALINEADA\n(quiebres sobre nodos)"),
]:
    for met, color in colores.items():
        sub = df[df["Método"] == met].sort_values("Δz")
        if len(sub):
            ax.loglog(sub["Δz"], sub["Error"], marcadores[met] + "-", color=color,
                      label=met, lw=1.5, ms=5, alpha=0.85, mfc="none", mew=1.4)

    # Rectas de referencia de pendiente p, ancladas al punto más grueso del trapecio
    ref = df[df["Método"] == "Trapecio"].sort_values("Δz")
    dz_ref = np.array([float(df["Δz"].min()), float(df["Δz"].max())])
    if len(ref):
        dz0, e0 = float(ref["Δz"].iloc[-1]), float(ref["Error"].iloc[-1])
        for p, estilo in [(2, ":"), (4, "--"), (6, "-.")]:
            ax.loglog(dz_ref, e0 * (dz_ref / dz0) ** p, estilo,
                      color="gray", lw=1.0, alpha=0.55)
            # Etiqueta al extremo izquierdo de cada recta, donde hay sitio libre
            ax.annotate(f"O(Δz^{p})", xy=(dz_ref[0], e0 * (dz_ref[0] / dz0) ** p),
                        xytext=(3, 2), textcoords="offset points",
                        fontsize=7.5, color="gray", va="bottom")

    ax.axhline(1e-13, color="crimson", ls="--", lw=1, alpha=0.55)
    ax.annotate("suelo de redondeo float64", xy=(dz_ref[0], 2e-13),
                fontsize=7.5, color="crimson", va="bottom")
    ax.set_xlabel("Δz  [cm]")
    ax.set_title(titulo, fontsize=11)
    ax.legend(fontsize=8, loc="upper left", framealpha=0.92)
    ax.grid(True, which="both", alpha=0.22)

# Señalamos explícitamente el hallazgo en cada panel
axes[0].annotate("las 4 curvas colapsan\na pendiente ≈ 2",
                 xy=(0.52, 0.40), xycoords="axes fraction", fontsize=9,
                 color="#444", ha="center",
                 bbox=dict(boxstyle="round,pad=0.4", fc="lightyellow",
                           ec="#bbb", alpha=0.95))
axes[1].annotate("Simpson y Boole: EXACTOS\n(pegados al suelo de redondeo)",
                 xy=(0.50, 0.12), xycoords="axes fraction", fontsize=9,
                 color="#444", ha="center",
                 bbox=dict(boxstyle="round,pad=0.4", fc="lightyellow",
                           ec="#bbb", alpha=0.95))

axes[0].set_ylabel("Error absoluto en el volumen  |V − V_exacto|  [cm³]")
axes[0].set_ylim(1e-17, 1e3)
fig.suptitle("Convergencia de los métodos de Newton-Cotes — "
             f"{ENVASE.nombre}  (V_exacto = {V_EXACTO:.4f} cm³)", fontsize=12)
fig.tight_layout()
plt.show()

print("\nLectura de la gráfica:")
print("  • Izquierda (malla ciega): las curvas de Simpson y Boole son MÁS PLANAS que")
print("    sus rectas de referencia → no alcanzan su orden teórico. Las esquinas del")
print("    envase les impiden aprovechar su orden alto.")
print("  • Derecha (malla alineada): las curvas caen mucho más rápido, y varias chocan")
print("    contra la línea roja del suelo de redondeo → error exacto salvo aritmética.")
print("  • La línea roja (1e-13) es el límite de float64. Por debajo, el 'error' ya no")
print("    es del método: es ruido de punto flotante acumulado en las sumas.")

**Cómo leer la gráfica:**

- **Eje X:** $\Delta z$ decreciente hacia la izquierda — más puntos, malla más fina.
- **Eje Y:** error absoluto en cm³, escala logarítmica.
- **Una curva más inclinada = método que mejora más rápido al refinar la malla.**
- **Panel izquierdo vs. derecho** — es el contraste que da sentido a toda la sección.
  A la izquierda, las curvas de Simpson 3/8 y Boole discurren casi paralelas a la del
  trapecio pese a ser teóricamente muy superiores. A la derecha se despegan y se hunden.
  **El mismo método, los mismos datos, y órdenes de magnitud de diferencia: lo único que
  cambió fue dónde se colocaron los nodos.**
- **La línea roja horizontal (1e-13)** es el suelo de redondeo. Cuando una curva llega
  ahí y se aplana, **ya no está midiendo el error del método**: está midiendo la
  acumulación de errores de redondeo de `float64` en la suma de miles de términos.
  Refinar más allá de ese punto **empeora** el resultado, porque se suman más términos y
  se acumula más redondeo, sin ganancia de aproximación. Es el equilibrio clásico entre
  **error de truncamiento** (baja al refinar) y **error de redondeo** (sube al refinar):
  existe un $\Delta z$ óptimo, y pasarse es contraproducente.

---
## 9. Representación gráfica — §8 del PDF

El enunciado pide *"representación gráfica del perfil y, opcionalmente, de la función
πr²"*. Hacemos las dos, más dos visualizaciones adicionales que ayudan a interpretar.

**Qué hace esta celda:** cuatro paneles.

1. **Perfil $r(z)$** con los nodos de la malla y los puntos de quiebre marcados. Es la
   silueta del envase que, en la fase de cámara, saldrá del contorno de la foto.
2. **Integrando $f(z) = \pi r(z)^2$** — el área del disco a cada altura. **El volumen es
   literalmente el área sombreada bajo esta curva.** Es la traducción visual de la
   integral.
3. **Sección longitudinal completa**, reflejando el perfil sobre el eje. Es la forma en
   que reconoces el envase a simple vista, y sirve para validar de un vistazo que las
   proporciones editadas en `TRAMOS` son las que pretendías.
4. **Aporte acumulado al volumen**: $V(z) = \pi\int_a^z r^2\,d\zeta$, normalizado. Dice
   **qué fracción del volumen total se ha acumulado a cada altura**, y por tanto dónde
   es crítico medir bien.

In [ ]:
z_fino = np.linspace(ENVASE.a, ENVASE.b, 1500)
r_fino = ENVASE.radio(z_fino)
f_fino = math.pi * r_fino ** 2
quiebres = ENVASE.puntos_de_quiebre()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- Panel 1: perfil r(z) ---------------------------------------------
ax = axes[0, 0]
ax.plot(z_fino, r_fino, "-", color="#1f77b4", lw=2, label="perfil r(z) continuo")
ax.plot(Z_BASE, R_BASE, "o", color="#d62728", ms=4.5,
        label=f"nodos de la malla (n={N_BASE})")
for q in quiebres:
    ax.axvline(q, color="gray", ls=":", lw=1.2, alpha=0.8)
ax.annotate("puntos de quiebre\n(no derivable)", xy=(quiebres[1], float(ENVASE.radio(quiebres[1])[0])),
            xytext=(quiebres[1] - 6.5, 2.0), fontsize=8, color="gray",
            arrowprops=dict(arrowstyle="->", color="gray", lw=0.9))
ax.set_xlabel("z  [cm]"); ax.set_ylabel("r(z)  [cm]")
ax.set_title("1. Perfil del envase — lo que dará el contorno de la foto", fontsize=10)
ax.legend(fontsize=8); ax.set_ylim(0, None)

# --- Panel 2: integrando pi*r^2 ---------------------------------------
ax = axes[0, 1]
# El área bajo π·r² es el VOLUMEN: [cm²]·[cm] = [cm³].
ax.fill_between(z_fino, 0, f_fino, alpha=0.28, color="#2ca02c",
                label=f"área bajo la curva = V = {V_EXACTO:.2f} cm³")
ax.plot(z_fino, f_fino, "-", color="#2ca02c", lw=2)
ax.plot(Z_BASE, math.pi * R_BASE ** 2, "o", color="#d62728", ms=4)
for q in quiebres:
    ax.axvline(q, color="gray", ls=":", lw=1.2, alpha=0.8)
ax.set_xlabel("z  [cm]"); ax.set_ylabel("f(z) = π·r(z)²  [cm²]")
ax.set_title("2. Integrando: área del disco a cada altura", fontsize=10)
ax.legend(fontsize=8)

# --- Panel 3: seccion longitudinal ------------------------------------
ax = axes[1, 0]
ax.fill_between(z_fino, -r_fino, r_fino, alpha=0.30, color="#1f77b4")
ax.plot(z_fino,  r_fino, "-", color="#1f77b4", lw=1.8)
ax.plot(z_fino, -r_fino, "-", color="#1f77b4", lw=1.8)
ax.axhline(0, color="#d62728", ls="--", lw=1.3, label="eje de revolución")
_r_max = float(r_fino.max())
for t in ENVASE.tramos:
    ax.axvline(t.z_fin, color="gray", ls=":", lw=0.9, alpha=0.7)
    # Solo etiquetamos los tramos con hueco suficiente; si no, se solapan.
    if t.altura >= 0.10 * ENVASE.altura:
        ax.text(0.5 * (t.z_ini + t.z_fin), _r_max * 1.18, t.nombre,
                ha="center", va="bottom", fontsize=7.5, color="dimgray")
ax.set_xlabel("z  [cm]"); ax.set_ylabel("radio  [cm]")
ax.set_title("3. Sección longitudinal del sólido de revolución", fontsize=10)
ax.legend(fontsize=8, loc="lower right")
ax.set_ylim(-_r_max * 1.45, _r_max * 1.55)
ax.set_aspect("equal", adjustable="box")

# --- Panel 4: volumen acumulado ---------------------------------------
ax = axes[1, 1]
V_acum = math.pi * integrate.cumulative_trapezoid(r_fino ** 2, z_fino, initial=0.0)
ax.plot(z_fino, 100 * V_acum / V_EXACTO, "-", color="#9467bd", lw=2.2)
for t in ENVASE.tramos:
    ax.axvline(t.z_fin, color="gray", ls=":", lw=0.9, alpha=0.7)
# Marcar el tramo que más aporta
_res = ENVASE.resumen()
_dom = _res.loc[_res["V tramo [cm³]"].idxmax()]
ax.axvspan(_dom["z_ini [cm]"], _dom["z_fin [cm]"], color="#9467bd", alpha=0.12)
ax.text(0.5 * (_dom["z_ini [cm]"] + _dom["z_fin [cm]"]), 45,
        f"{_dom['tramo']}\n{_dom['% del total']:.1f} % del volumen",
        ha="center", fontsize=8.5, color="#5b3a8e", weight="bold")
ax.set_xlabel("z  [cm]"); ax.set_ylabel("volumen acumulado  [% del total]")
ax.set_title("4. ¿Dónde está el volumen? — dónde hay que medir bien", fontsize=10)
ax.set_ylim(0, 105)

fig.suptitle(f"{ENVASE.nombre} — V exacto = {V_EXACTO:.4f} cm³ = {V_EXACTO:.2f} mL",
             fontsize=12.5)
fig.tight_layout()
plt.show()

**Cómo leer las gráficas:**

- **Panel 1** — los puntos rojos son la tabla de datos; la línea azul es el envase real.
  La **distancia entre puntos rojos es Δz**. Si imaginas que cada punto rojo es una fila
  de píxeles, estás viendo exactamente lo que entregará la fase de cámara. Las líneas
  punteadas verticales son los quiebres: observa que en general **no coinciden con
  puntos rojos** — esa desalineación es la causa de todo lo visto en la Sección 8.
- **Panel 2** — el volumen **es** el área verde sombreada. Nota el escalón brusco en el
  hombro: $\pi r^2$ se desploma de ~36 a ~5.7 cm² en apenas 3.5 cm. Esa caída abrupta es
  la zona donde los métodos numéricos sufren.
- **Panel 3** — la silueta reconocible. Úsalo como control visual tras editar `TRAMOS`:
  si no parece un envase, las proporciones están mal.
- **Panel 4 — el panel con la moraleja práctica.** La curva es empinada donde el envase
  es ancho y casi plana en el cuello. La banda sombreada marca el tramo dominante. Como
  el cuerpo cilíndrico aporta **~82 %** del volumen, la precisión del resultado final
  depende sobre todo de medir bien **el radio del cuerpo**. Un error de 1 mm ahí cuesta
  ~18 cm³ (3 % del total); el mismo error en el cuello cuesta menos de 0.5 cm³. En la
  prueba de campo, esto dice **dónde enfocar la cámara y dónde poner la referencia de
  escala**.

---
## 9.1 Análisis de sensibilidad: por qué la escala manda sobre el método

**Qué hace esta celda:** cuantifica y compara las dos fuentes de error que competirán en
el proyecto real.

1. **Error del método numérico** — el que acabamos de medir, con datos perfectos.
2. **Error de escala** — el que aparecerá al convertir píxeles a centímetros.

Simulamos un error de calibración introduciendo un factor $s$ en todos los radios:
$r \to (1+s)\,r$. Como $V \propto r^2$:

$$\frac{V(s)}{V} = (1+s)^2 \;\approx\; 1 + 2s$$

**Este es el argumento que cierra el diseño del proyecto**, y conviene tenerlo con
números concretos antes de escribir una sola línea de código de visión por computador.

In [ ]:
errores_escala = np.array([0.001, 0.005, 0.01, 0.02, 0.025, 0.05])
filas_sens = []
for s in errores_escala:
    # Reutilizamos nuestra propia regla del trapecio sobre la malla fina:
    # coherente con el resto del notebook y sin dependencias de versión de numpy.
    V_s = regla_trapecio(z_fino, (1 + s) * r_fino).volumen_cm3
    filas_sens.append({
        "Error de escala [%]": 100 * s,
        "Factor (1+s)²": (1 + s) ** 2,
        "V estimado [cm³]": V_s,
        "Error en V [cm³]": V_s - V_EXACTO,
        "Error en V [%]": 100 * (V_s - V_EXACTO) / V_EXACTO,
        "¿Pasa el 5 %?": "✓ sí" if abs(100 * (V_s - V_EXACTO) / V_EXACTO) < 5 else "✗ NO",
    })
SENS = pd.DataFrame(filas_sens)

print("=" * 78)
print("SENSIBILIDAD AL ERROR DE ESCALA  (el radio entra al CUADRADO)")
print("=" * 78)
display(SENS)

# Separamos las Newton-Cotes de Gauss: son regímenes muy distintos.
_nc = TABLA_COMP[TABLA_COMP["Familia"].str.contains("Newton-Cotes")]
_err_nc_mejor = _nc["Err. rel. [%]"].min()
_err_nc_peor = _nc["Err. rel. [%]"].max()
_err_escala_1 = 100 * ((1.01) ** 2 - 1)

print("=" * 78)
print("COMPARACIÓN DE LAS DOS FUENTES DE ERROR")
print("-" * 78)
print(f"  (A) Error del MÉTODO numérico, con n = {N_BASE} y datos PERFECTOS:")
print(f"        mejor Newton-Cotes : {_err_nc_mejor:.3e} %")
print(f"        peor  Newton-Cotes : {_err_nc_peor:.3e} %")
print(f"  (B) Error por ESCALA mal calibrada al 1 %:")
print(f"        resulta            : {_err_escala_1:.3f} %")
print("-" * 78)
print(f"  Un error de escala de apenas 1 % pesa:")
print(f"     {_err_escala_1 / _err_nc_peor:,.0f}× más que el PEOR Newton-Cotes")
print(f"     {_err_escala_1 / _err_nc_mejor:,.0f}× más que el MEJOR Newton-Cotes")
print("-" * 78)
print(f"  Y el error del método se puede reducir GRATIS aumentando n:")
for _n in (60, 240, 1200):
    _zt, _rt = muestrear_perfil(ENVASE, _n)
    _e = 100 * abs(regla_trapecio(_zt, _rt).volumen_cm3 - V_EXACTO) / V_EXACTO
    print(f"     n = {_n:>5}  →  trapecio con {_e:.3e} %   "
          f"({_err_escala_1 / _e:,.0f}× menor que el error de escala del 1 %)")
print("=" * 78)
print("  → El presupuesto de error del 5 % se gasta casi íntegramente en la")
print("    calibración píxel→cm y en la extracción del contorno, NO en la cuadratura.")
print("    Escala con error < 2.5 %  ⇒  volumen con error < 5 %.")
print("=" * 78)

**Cómo leer el resultado:**

- **La columna `Error en V [%]` es aproximadamente el doble de la columna
  `Error de escala [%]`.** Es la propagación $\Delta V/V = 2\,\Delta r/r$ visible en
  datos, no en una fórmula.
- **La fila del 2.5 % es la frontera:** ahí el error de volumen llega justo al 5 %. Ese
  es el **presupuesto de calibración** del proyecto: la escala píxel→cm debe tener un
  error menor que 2.5 %, y con margen de seguridad, menor que 1 %.
- **Los multiplicadores** son la conclusión cuantitativa del notebook. Con $n = 60$, un
  error de escala del 1 % pesa unas **40 veces más** que el peor Newton-Cotes y unas
  **200 veces más** que el mejor. Y el bloque final muestra algo todavía más decisivo:
  **el error del método se reduce gratis subiendo $n$** — con $n = 1200$ el trapecio baja
  a ~3e-5 %, unas 76 000 veces por debajo del error de escala.
- **Por qué esa asimetría lo decide todo:** subir $n$ no cuesta nada (son microsegundos
  de CPU y la foto ya tiene esos píxeles). Mejorar la calibración de la escala cuesta
  imprimir bien un marcador, medirlo con regla, montar la escena y controlar la
  perspectiva. **Una de las dos fuentes de error es gratis de eliminar y la otra no**, y
  resulta ser justo la cara la que domina.

> **Qué implica para lo que viene.** El esfuerzo de la fase 2 debe concentrarse en:
> (a) que el marcador de referencia esté **medido con precisión real** — imprimirlo y
> verificar con regla, porque las impresoras escalan sin avisar; (b) que esté
> **coplanar con el eje del envase**; (c) que la foto se tome **de lejos con zoom**,
> para aproximar la proyección ortográfica que supone nuestro modelo; y (d) que el
> contorno se extraiga limpio. El motor de integración ya está resuelto y sobra de
> precisión.

---
## 10. Exportación de datos y resultados — §8 del PDF

El enunciado pide *"exportación o posibilidad de copiar los datos/resultados para el
informe"*.

**Qué hace esta celda:** genera cuatro archivos y los ofrece para descarga:

| Archivo | Contenido |
|---|---|
| `perfil_datos.csv` | la tabla $(i, z_i, r_i, r_i^2, \pi r_i^2)$ — §7 Etapa 4 |
| `comparacion_metodos.csv` | la tabla comparativa completa — §7 Etapa 5 y 6 |
| `convergencia.csv` | el barrido de errores frente a $\Delta z$ — §6 |
| `resumen_informe.md` | un resumen en Markdown, listo para pegar |

En Colab aparecerán en el panel de archivos (📁 a la izquierda) y se dispara la descarga
automática. Fuera de Colab, se escriben en el directorio de trabajo.

In [ ]:
import io, os, json

def _a_markdown(df: pd.DataFrame, floatfmt: str = ".4f") -> str:
    """DataFrame -> tabla Markdown, con respaldo si falta el paquete `tabulate`.

    pandas.DataFrame.to_markdown() delega en `tabulate`, que no siempre está
    instalado. En vez de dejar que la exportación falle por una dependencia
    opcional, generamos la tabla a mano. El objetivo de esta celda es entregar
    archivos al usuario; no puede caerse por un detalle de empaquetado.
    """
    try:
        return df.to_markdown(index=False, floatfmt=floatfmt)
    except (ImportError, AttributeError):
        def _fmt(v):
            if isinstance(v, float):
                return f"{v:{floatfmt}}"
            return str(v)
        cabecera = "| " + " | ".join(str(c) for c in df.columns) + " |"
        separador = "|" + "|".join("---" for _ in df.columns) + "|"
        filas = ["| " + " | ".join(_fmt(v) for v in fila) + " |"
                 for fila in df.itertuples(index=False, name=None)]
        return "\n".join([cabecera, separador, *filas])


ARCHIVOS: dict[str, str] = {}

# --- 1. Tabla de datos del perfil --------------------------------------
TABLA_BASE.to_csv("perfil_datos.csv", index=False, encoding="utf-8-sig")
ARCHIVOS["perfil_datos.csv"] = f"{len(TABLA_BASE)} filas — tabla (z, r, r², πr²)"

# --- 2. Comparación de métodos -----------------------------------------
TABLA_COMP.to_csv("comparacion_metodos.csv", index=False, encoding="utf-8-sig")
ARCHIVOS["comparacion_metodos.csv"] = f"{len(TABLA_COMP)} métodos con todos sus parámetros"

# --- 3. Estudio de convergencia ----------------------------------------
_conv = pd.concat([
    CONV_CIEGA.assign(Malla="ciega"),
    CONV_ALIN.rename(columns={"n_total": "n"}).assign(Malla="alineada"),
], ignore_index=True)
_conv.to_csv("convergencia.csv", index=False, encoding="utf-8-sig")
ARCHIVOS["convergencia.csv"] = f"{len(_conv)} puntos de convergencia (ambas mallas)"

# --- 4. Resumen en Markdown --------------------------------------------
_buf = io.StringIO()
_buf.write(f"# Resultados — {ENVASE.nombre}\n\n")
_buf.write("## Parámetros del envase\n\n")
_buf.write(f"- Intervalo de integración: [{ENVASE.a:.4f}, {ENVASE.b:.4f}] cm\n")
_buf.write(f"- Altura total: {ENVASE.altura:.4f} cm\n")
_buf.write(f"- Puntos de quiebre: {[round(float(q), 4) for q in quiebres]} cm\n")
_buf.write(f"- Volumen exacto de referencia: **{V_EXACTO:.6f} cm³ = {V_EXACTO:.2f} mL**\n")
_buf.write(f"- Integral exacta ∫r²dz: {I_EXACTA:.6f} cm³\n\n")
_buf.write("### Composición por tramos\n\n")
_buf.write(_a_markdown(ENVASE.resumen(), ".4f"))
_buf.write("\n\n## Discretización\n\n")
_buf.write(f"- n (subintervalos): {N_BASE}\n")
_buf.write(f"- Puntos: {N_BASE + 1}\n")
_buf.write(f"- **Δz = (b − a)/n = {DZ_BASE:.6f} cm**\n")
_buf.write(f"- Malla uniforme; n múltiplo de 12 (compatible con los cuatro métodos)\n\n")
_buf.write("## Comparación de métodos\n\n")
_buf.write(_a_markdown(TABLA_COMP[["Método", "n", "Evals", "Δz [cm]",
                                   "Orden / exactitud", "I = ∫r²dz [cm³]",
                                   "V = πI [cm³]", "Err. abs. [cm³]",
                                   "Err. rel. [%]", "Tiempo [ms]"]], ".6g"))
_buf.write("\n\n## Órdenes de convergencia observados\n\n")
_buf.write("| Método | Teórico | Malla ciega | Malla alineada |\n|---|---|---|---|\n")
def _fmt_orden(v: float) -> str:
    """NaN aquí significa 'exacto salvo redondeo', no 'no se pudo calcular'."""
    return "exacto (suelo de redondeo)" if math.isnan(v) else f"{v:.2f}"

for met, p_teo in TEORICO.items():
    _buf.write(f"| {met} | {p_teo} | {_fmt_orden(orden_observado(CONV_CIEGA, met))} | "
               f"{_fmt_orden(orden_observado(CONV_ALIN, met))} |\n")
_buf.write("\n## Sensibilidad al error de escala\n\n")
_buf.write(_a_markdown(SENS, ".4f"))
_buf.write("\n")

with open("resumen_informe.md", "w", encoding="utf-8") as fh:
    fh.write(_buf.getvalue())
ARCHIVOS["resumen_informe.md"] = "resumen en Markdown listo para el informe"

# --- Reporte y descarga ------------------------------------------------
print("=" * 74)
print("ARCHIVOS GENERADOS")
print("=" * 74)
for nombre, desc in ARCHIVOS.items():
    print(f"  ✓ {nombre:<28} {os.path.getsize(nombre):>8,} bytes   {desc}")
print("=" * 74)

try:
    from google.colab import files as _colab_files
    print("\nEntorno Colab detectado — iniciando descarga...")
    for nombre in ARCHIVOS:
        _colab_files.download(nombre)
except ImportError:
    print(f"\nFuera de Colab: los archivos están en {os.getcwd()}")

**Cómo leer el resultado:** las cuatro marcas ✓ con sus tamaños confirman la
escritura. En Colab se dispara la descarga de cada archivo.

`resumen_informe.md` está pensado para pegarse directamente en el documento del
laboratorio: contiene los parámetros del envase, la discretización con $\Delta z$
explícito, la tabla comparativa, los órdenes de convergencia y el análisis de
sensibilidad, todo en tablas Markdown ya formateadas.

---
## 11. Fuentes de error y discusión — §9 MVP

El MVP exige *"explicar las principales fuentes de error"*. Con lo medido en este
notebook podemos ordenarlas **por magnitud real**, no por intuición.

### 11.1 Inventario de fuentes de error

| # | Fuente | Magnitud típica en $V$ | ¿Medida aquí? | Cómo se ataca |
|---|---|---|---|---|
| 1 | **Calibración de escala px→cm** | **2× el error de escala** | Simulada (§9.1) | Marcador ArUco medido con regla; verificar la impresión |
| 2 | **Proyección perspectiva** | 2·(R/d), ~3-7 % a 1 m | No (fase 2) | Cámara lejos + zoom; homografía coplanar |
| 3 | **Extracción del contorno** | 1-3 px por borde | No (fase 2) | Segmentación robusta; comparar Otsu vs GrabCut |
| 4 | **Distorsión de lente** | 2-5 % en los bordes | No (fase 2) | Calibrar con ChArUco y aplicar `undistort` |
| 5 | **Asimetría del envase** | 0-5 % según el envase | No (fase 2) | Promediar radios izquierdo y derecho; reportar la asimetría |
| 6 | **Discretización ($\Delta z$)** | **0.01–0.05 %** con n=60 | **Sí (§7, §8)** | Subir $n$ (gratis), o alinear la malla con los quiebres |
| 7 | **Elección del método (Newton-Cotes)** | **0.01–0.05 %** | **Sí (§7)** | Indiferente entre los cuatro |
| 7b | **Gauss de bajo orden** | **0.6 % con m=5; >5 % con m=2,3** | **Sí (§5.7)** | ⚠️ No usar como método principal |
| 8 | **Redondeo `float64`** | ~1e-14 % | **Sí (§8)** | Irrelevante; es el suelo de la aritmética |

### 11.2 La conclusión que ordena el resto del proyecto

Las fuentes **6, 7 y 8 son despreciables**: consumen menos del 1 % del margen del 5 %, y
además **el error de discretización se reduce gratis subiendo $n$** (con $n=1200$ el
trapecio baja a ~3e-5 %).

La fuente **7b es la única excepción numérica y es una decisión de diseño, no un error
inevitable**: Gauss-Legendre global de bajo orden falla el criterio del 5 % por sí solo,
incluso con datos perfectos. Se reporta porque el enunciado lo exige, pero no se usa como
valor de salida.

Todo el presupuesto de error se lo van a llevar las fuentes **1 a 5**, que son de
**adquisición y procesamiento de imagen**. El motor de integración está resuelto y le
sobra precisión.

Esto no le resta valor al laboratorio: se lo da. Un ingeniero que mide dónde está el
error antes de optimizar es un ingeniero que no desperdicia esfuerzo. La respuesta a la
pregunta orientadora del enunciado —

> *¿Cómo convertir una imagen capturada por un celular en datos numéricos confiables
> para estimar un volumen mediante integración numérica?* —

es, a la luz de estos resultados: **el reto no está en la integración; está en el
"confiables"**.

### 11.3 Decisiones cerradas para la fase 2

| Aspecto | Decisión | Justificación |
|---|---|---|
| **Método a producción** | Simpson 1/3, con trapecio de respaldo | Mejor relación precisión/robustez; el trapecio no tiene restricciones sobre $n$ |
| **Estrategia de malla** | Remuestrear a $n$ múltiplo de 12 e integrar por tramos si se detectan quiebres | Permite ejecutar los cuatro métodos y recupera el orden teórico |
| **Gauss** | Se reporta, pero sobre spline; no se usa como valor principal | Su ventaja teórica no sobrevive a datos ruidosos con interpolación |
| **Validación** | Volumen **exterior** por desplazamiento de agua | Única cantidad físicamente comparable con una silueta |
| **Presupuesto de escala** | Error < 1 % (objetivo), < 2.5 % (límite) | $\Delta V/V = 2\,\Delta r/r$, medido en §9.1 |

### 11.4 Qué falta — hoja de ruta

1. **Escala:** detección ArUco + homografía, con dos clics manuales como respaldo.
2. **Extracción del contorno:** Otsu/Canny y GrabCut, comparados (cubre el requisito de
   la Etapa 1 de investigar al menos dos estrategias).
3. **Eje de revolución:** detección por centroide con regresión robusta; corrección de
   inclinación.
4. **Perfil:** radio como semiancho por fila, promediando ambos lados; **reportar la
   asimetría como indicador de calidad de la medición**.
5. **Remuestreo:** a malla uniforme con $n$ múltiplo de 12, por spline.
6. **Integración:** este motor, sin cambios.
7. **Aplicación Streamlit:** carga de imagen, `st.camera_input()`, y toda la trazabilidad
   que ya produce este notebook.
8. **Validación de campo:** desplazamiento de agua sobre 3-5 envases reales.

---

### Cierre

El enunciado dice que el reto *"no consiste únicamente en calcular un volumen, consiste
en demostrar cómo aplicas todo lo que has aprendido en tu disciplina para resolver retos
de ingeniería"*.

La contribución de ingeniería de esta fase no es haber implementado cuatro fórmulas de
cuadratura — eso es la parte fácil, y lo hemos demostrado midiendo que su error es
despreciable. Es haber **medido el presupuesto de error antes de construir**, y haber
salido con un número concreto: *la escala debe ser mejor que el 2.5 %*. Ese número es lo
que va a dirigir todas las decisiones de la fase 2.